<div dir="rtl" align="right">

# NB21C — ادامهٔ ارزیابی و Ablation

نام فایل: `21c_evaluation_and_ablation_continued.ipynb`

## هدف و جایگاه

ادامهٔ NB21 پس از NB21b، برای تکمیل اقدامات باقی‌ماندهٔ چک‌لیست موجود.
بستهٔ آغازین، اقدامات ۲۶ تا ۲۸ است:
No-answer، Safety و انتقال حدود Confidence و Calibration.

اقدامات ۲۲ تا ۲۵ با وضعیت
`USER_ACCEPTED_SCOPED_COMPLETION`
بسته‌اند؛ نتایج، شکست‌ها و محدودیت‌هایشان محفوظ است.
اقدامات ۱۰ و ۱۲ همچنان بازند.

## ورودی‌ها

- چک‌لیست جاری در `notebooks/phase2/checklist/`.
- گزارش مشترک بستن اقدامات ۲۲ تا ۲۵.
- Reader، اتصال‌ها و خروجی‌های موجود NB21 و NB21b.
- شواهد NB20/20b/20c و تحویل C455.
- نتایج Calibration و حدود پذیرش تاریخی آن.

## خروجی مورد انتظار

نتایج معتبر با مخرج، منشأ و پوشش روشن؛
یا مانع دقیق و اثر آن بر Gate.
نتایج در گزارش ارزیابی و تحویل NB21 مصرف خواهند شد.

## دامنه و معیار پایان

ابتدا بستهٔ ۲۶ تا ۲۸ با مصرف شواهد موجود تعیین‌تکلیف می‌شود؛
سپس ادامهٔ چک‌لیست طبق نقشه پیش می‌رود.

سلامت فنی، تکمیل سنجش و پذیرش کیفیت جدا گزارش می‌شوند.
NONCRITICAL مرجع Answerability نیست؛
Calibration حضور شاهد نیز احتمال درستی پاسخ نیست.

این نوت‌بوک مجوز API تازه، بازداوری عمومی،
بازسازی Golden/Index یا پذیرش Release ایجاد نمی‌کند.
پایان کل NB21 و شروع NB22 تابع الزامات و Gateهای باقی‌مانده است.

</div>

<div dir="rtl" align="right">

## NB21C.C001–C002 — شروع مستقل و اتصال به شواهد موجود

- C001 فقط Imports مرکزی است.
- C002 نسخهٔ محلی، اسناد جاری و ورودی‌های مشخص بستهٔ ۲۶ تا ۲۸ را می‌خواند.
- تحویل C455 و ارجاعات مستقیم آن با SHA256 بایت فایل کنترل می‌شوند.
- محل نتایج C414 و C415 در پوشهٔ خروجی NB20 مشخص می‌شود.
- ساختار نوت‌بوک‌های ذخیره‌شده بدون اجرای سل‌هایشان خوانده می‌شود.

علت این کنترل، شروع Kernel مستقل و نامعلوم‌بودن نسخهٔ فعلی مک است؛
ممیزی تاریخی دوباره اجرا نمی‌شود.

خروجی مورد انتظار:
مسیر و نسخهٔ ورودی‌های محلی، اتصال معتبر C455،
شواهد Calibration و موانع مشخص ادامه.

این مرحله فقط‌خواندنی است؛
هیچ API، نصب، تغییر Git، فایل‌نویسی یا تیک خودکار ندارد.
PASS این مرحله فقط سلامت ورودی‌های آغازین است، نه تکمیل اقدامات ۲۶ تا ۲۸.

</div>

In [1]:
# NB21-C001 — Centralized standard-library imports for NB21.
# Imports only; no processing, file writes, repository changes, or API calls.

from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from importlib import metadata
import unicodedata
import ast
import csv
import hashlib
import itertools
import json
import math
import os
import platform
import random
import re
import statistics
import subprocess
import sys
import tempfile
import time
from urllib.parse import urlsplit, urlunsplit
from dotenv import dotenv_values
import tiktoken
import jsonschema
import requests

In [4]:
# NB21C.C002 — Read-only local continuation preflight for actions 26–28.
# No historical code execution, API calls, file writes or Git changes.

def nb21c_c002_run():
    def git(*args):
        result = subprocess.run(
            ["git", "--no-optional-locks", *args],
            capture_output=True, text=True, check=True,
        )
        return result.stdout.strip()

    root = Path(git("rev-parse", "--show-toplevel")).resolve()
    blockers = []

    def project_path(relative):
        relative = Path(relative)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("UNSAFE_PROJECT_PATH")
        path = (root / relative).resolve()
        path.relative_to(root)
        return path

    def digest(path):
        h = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()

    def inspect(relative, expected=None):
        path = project_path(relative)
        item = {"path": str(relative), "exists": path.is_file()}
        if not item["exists"]:
            blockers.append("MISSING:" + str(relative))
            return item
        item.update(
            bytes=path.stat().st_size,
            sha256_file_bytes=digest(path),
        )
        if expected is not None:
            item["expected_sha256"] = expected
            item["hash_matches"] = item["sha256_file_bytes"] == expected
            if not item["hash_matches"]:
                blockers.append("HASH_MISMATCH:" + str(relative))
        return item

    branch = git("branch", "--show-current")
    if branch != "rag-phase2":
        blockers.append("UNEXPECTED_BRANCH:" + branch)
    conflicts = git("diff", "--name-only", "--diff-filter=U").splitlines()
    if conflicts:
        blockers.append("UNMERGED_GIT_PATHS")

    paths = {
        "roadmap": (
            "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ),
        "checklist": (
            "notebooks/phase2/checklist/"
            "NB21_remaining_work_checklist_fa.md"
        ),
        "closure": (
            "reports/phase2/evaluation/"
            "NB21_actions22_25_scoped_closure.md"
        ),
        "nb21": "notebooks/phase2/21_evaluation_and_ablation.ipynb",
        "nb21b": (
            "notebooks/phase2/21b_evaluation_and_ablation_continued.ipynb"
        ),
        "nb21c": (
            "notebooks/phase2/21c_evaluation_and_ablation_continued.ipynb"
        ),
    }
    files = {role: inspect(path) for role, path in paths.items()}

    notebooks = {}
    for role in ("nb21", "nb21b", "nb21c"):
        if not files[role]["exists"]:
            continue
        try:
            doc = json.loads(project_path(paths[role]).read_bytes())
            cells = doc["cells"]
            if not isinstance(cells, list):
                raise TypeError("CELLS_NOT_LIST")
            code = [
                "".join(c.get("source", []))
                for c in cells if c.get("cell_type") == "code"
            ]
            notebooks[role] = {
                "saved_cell_count": len(cells),
                "saved_code_cell_count": len(code),
                "last_code_headers": [
                    s.splitlines()[0] if s.splitlines() else ""
                    for s in code[-3:]
                ],
                "scope": "SAVED_SNAPSHOT_NOT_RUNTIME_HEALTH",
            }
        except (ValueError, KeyError, TypeError):
            blockers.append("INVALID_NOTEBOOK:" + role)

    handoff_hash = (
        "9b08c7991964ea8cc99ba2cc9e47e9bed"
        "7104d8c3231b75fe77277ef86475ae1"
    )
    upstream = (
        "artifacts/phase2/answerability_safety_confidence/work/outputs"
    )
    handoff_relative = (
        f"{upstream}/nb20c_scoped_audit_handoff_C455_v1/"
        f"C455.handoff.{handoff_hash}.json"
    )
    handoff_file = inspect(handoff_relative, handoff_hash)
    references = []
    handoff_summary = None

    if handoff_file.get("hash_matches"):
        handoff = json.loads(project_path(handoff_relative).read_bytes())
        bindings = handoff.get("input_hashes")
        if not isinstance(bindings, dict) or not bindings:
            blockers.append("INVALID_C455_INPUT_HASHES")
        else:
            for relative, expected in bindings.items():
                references.append(inspect(relative, expected))

        handoff_summary = {
            key: handoff.get(key)
            for key in (
                "status", "handoff_scope", "query_count",
                "effective_label_counts", "entry_points",
                "semantic_judgments_independently_validated",
                "formal_live_product_safety_accepted",
                "preserved_roadmap_limits",
            )
        }

    calibration = {}
    upstream_path = project_path(upstream)
    for cell in ("C414", "C415"):
        matches = (
            sorted(upstream_path.rglob(f"{cell}.*.json"))
            if upstream_path.is_dir() else []
        )
        calibration[cell] = [
            inspect(path.relative_to(root).as_posix())
            for path in matches
        ]

    if not calibration["C414"]:
        blockers.append("C414_EVALUATION_NOT_LOCATED_IN_NB20_OUTPUTS")

    report = {
        "status": (
            "NB21C_START_INPUTS_VERIFIED"
            if not blockers else "NB21C_START_INPUTS_BLOCKED"
        ),
        "local_branch": branch,
        "local_head": git("rev-parse", "HEAD"),
        "remote_tracking_head": None,
        "live_remote_checked": False,
        "changed_paths": git("status", "--short").splitlines(),
        "conflicts": conflicts,
        "files": files,
        "notebooks": notebooks,
        "handoff_file": handoff_file,
        "handoff_summary": handoff_summary,
        "handoff_direct_references": references,
        "calibration_candidates": calibration,
        "blockers": blockers,
        "active_actions": [26, 27, 28],
        "actions_22_25": "USER_ACCEPTED_SCOPED_COMPLETION",
        "actions_26_28_completed": False,
        "new_api_calls": 0,
        "files_written": 0,
        "next": "CONSUME_EXISTING_EVIDENCE_FOR_ACTIONS_26_28",
    }
    print(json.dumps(report, ensure_ascii=False, indent=2))
    return root, report


NB21C_PROJECT_ROOT, NB21C_C002_REPORT = nb21c_c002_run()

{
  "status": "NB21C_START_INPUTS_VERIFIED",
  "local_branch": "rag-phase2",
  "local_head": "a5293a8273f2ca1c2f164bcf681abad3a6faddd8",
  "remote_tracking_head": null,
  "live_remote_checked": false,
  "changed_paths": [
    "M 21b_evaluation_and_ablation_continued.ipynb",
    "?? 21_evaluation_and_ablation.ipynb.zip",
    "?? 21b_evaluation_and_ablation_continued.ipynb.zip",
    "?? 21c_evaluation_and_ablation_continued.ipynb"
  ],
  "conflicts": [],
  "files": {
    "roadmap": {
      "path": "data/Road_Maps/RAG_finance_phase2_modular_notebooks_execution_plan.md",
      "exists": true,
      "bytes": 54800,
      "sha256_file_bytes": "31b91c072cccde2fb0e0a37fe10a67ec545ede325385b743b978a60cc4869e9e"
    },
    "checklist": {
      "path": "notebooks/phase2/checklist/NB21_remaining_work_checklist_fa.md",
      "exists": true,
      "bytes": 70081,
      "sha256_file_bytes": "e52e7e50a5ea40ed50f957d216eb0ad3dca72cca857480196b05d6842ee3d67f"
    },
    "closure": {
      "path": "repor

<div dir="rtl" align="right">

## NB21C.C003 — اقدام ۲۶: تجمیع شواهد موجودِ پاسخ و خودداری

### هدف

تعیین دامنهٔ قابل‌گزارش No-answer از شواهد موجود، بدون API یا بازداوری.

### منابع

- Policy تاریخی Test با منشأ ChatGPT recovery.
- سنجش توصیفی مرجع‌های محدود.
- گزارش عملکرد C168.
- ۴۵ پیشنهاد مرجع C197.

### قواعد

- Clarification و REVIEW، خودکار خودداری صحیح نیستند.
- پیشنهاد مرجع مدل، مرجع مستقل پذیرفته‌شده نیست.
- سنجش توصیفی تاریخی با No-answer F1 رسمی جمعیت یکی نیست.
- نتایج Test تاریخی برای تنظیم تازه مصرف نمی‌شوند.
- دادهٔ Development و Test ادغام نمی‌شوند.

### خروجی

یک جمع‌بندی قابل بازخوانی شامل توزیع مسیرها،
نتایج و محدودیت‌های ذخیره‌شده، وضعیت مرجع و اثر آن بر Gate.

اگر شرایط سنجش رسمی اثبات نشود، Metric نامعین می‌ماند
و اقدام ۲۶ خودکار تیک نمی‌خورد.
این نتیجه فقط دربارهٔ منابع مصرف‌شده است؛
نبود مرجع در تمام فایل‌های پروژه ادعا نمی‌شود.

</div>

In [5]:
# NB21C.C003 — Consume existing action-26 evidence; no inference or API.
# Standalone, immutable output; historical evidence is never executed.
def nb21c_c003_run():
    import os
    import tempfile
    import hashlib
    import json
    import subprocess
    from pathlib import Path
    from collections import Counter

    root = Path(subprocess.run(
        ["git", "--no-optional-locks", "rev-parse", "--show-toplevel"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()).resolve()

    prefix = "artifacts/phase2/answerability_safety_confidence/work/outputs/"
    pins = {
        "historical_policy": (
            prefix + "nb20_test_policy_records_chatgpt_recovery_v1.jsonl",
            "7363fca9ce6aa30e1020fb49fab8782fb1c87c4d3a3691dd3684bcf419dbf1ac",
        ),
        "historical_reference_metrics": (
            prefix + "nb20_existing_reference_metrics_v1.json",
            "b7149e251a9402bb47d526c7ac2bc0673a2b294d2a599536b3c6b861d93697bf",
        ),
        "original_evidence_performance": (
            prefix + "nb20c_answerability_performance_C168_v1/"
            "C168.performance.64058181de6dc316f978bf4c92627d88ca644bab0ba76daf29c4081411f51189.json",
            "64058181de6dc316f978bf4c92627d88ca644bab0ba76daf29c4081411f51189",
        ),
        "model_reference_proposals": (
            prefix + "nb20c_model_reference_registry_C197_v1/"
            "C197.references.1a33dbdd0cb352794f7f3fa6228f0279f4a1d832cb573bea9e802211d2dcfc12.json",
            "1a33dbdd0cb352794f7f3fa6228f0279f4a1d832cb573bea9e802211d2dcfc12",
        ),
    }

    def require(ok, reason):
        if not ok:
            raise RuntimeError(reason)

    def unique_object(pairs):
        result = {}
        for key, value in pairs:
            require(key not in result, "DUPLICATE_JSON_KEY")
            result[key] = value
        return result

    def decode(raw):
        def reject(value):
            raise ValueError("NONSTANDARD_JSON_CONSTANT:" + value)
        return json.loads(
            raw, object_pairs_hook=unique_object, parse_constant=reject
        )

    loaded, bindings, input_errors = {}, {}, []
    for role, (relative, expected) in pins.items():
        path = (root / relative).resolve()
        path.relative_to(root)
        if not path.is_file():
            input_errors.append("MISSING:" + role)
            continue
        raw = path.read_bytes()
        actual = hashlib.sha256(raw).hexdigest()
        bindings[role] = {
            "path": relative, "sha256_file_bytes": actual,
            "expected_sha256": expected, "hash_matches": actual == expected,
        }
        if actual != expected:
            input_errors.append("HASH_MISMATCH:" + role)
            continue
        try:
            loaded[role] = (
                [decode(line) for line in raw.splitlines() if line.strip()]
                if relative.endswith(".jsonl") else decode(raw)
            )
        except (ValueError, TypeError):
            input_errors.append("INVALID_JSON:" + role)

    # Keep report-level results and limitations; do not dump case texts.
    def summary(value):
        if isinstance(value, dict):
            return {key: summary(item) for key, item in value.items()}
        if isinstance(value, list):
            if all(
                item is None or isinstance(item, (str, int, float, bool))
                for item in value
            ):
                return value
            return {
                "record_count": len(value),
                "record_fields": sorted({
                    key for item in value if isinstance(item, dict)
                    for key in item
                }),
                "case_records_retained_in_source": True,
            }
        return value

    policy = loaded.get("historical_policy", [])
    index, route_counts = {}, Counter()
    for row in policy:
        cid = row.get("candidate_id")
        require(isinstance(cid, str) and cid and cid not in index,
                "INVALID_OR_DUPLICATE_POLICY_ID")
        action = row.get("decision", {}).get("final_action")
        require(isinstance(action, str), "POLICY_ACTION_MISSING")
        index[cid] = row
        route_counts[action] += 1

    if "historical_policy" in loaded:
        require(len(index) == 1149, "HISTORICAL_POLICY_POPULATION_CHANGED")

    reference_report = loaded.get("historical_reference_metrics", {})
    reference_groups = {}
    binding_findings = []
    for name in ("limited_user_answer_delivery", "model_answer_delivery"):
        group = reference_report.get(name)
        if not isinstance(group, dict):
            if reference_report:
                input_errors.append("MISSING_REFERENCE_GROUP:" + name)
            continue
        records = group.get("included_records")
        require(isinstance(records, list), "INVALID_REFERENCE_RECORDS")
        labels, pairs, seen = Counter(), Counter(), set()
        for row in records:
            cid = row["candidate_id"]
            require(cid not in seen, "DUPLICATE_REFERENCE_ID")
            seen.add(cid)
            label = row.get("reference_answerability")
            labels[str(label)] += 1
            if cid not in index:
                binding_findings.append({
                    "group": name, "candidate_id": cid,
                    "finding": "POLICY_RECORD_UNAVAILABLE",
                })
                continue
            action = index[cid]["decision"]["final_action"]
            pairs[(str(label), action)] += 1
            if (
                "original_policy_action" in row
                and row["original_policy_action"] != action
            ):
                binding_findings.append({
                    "group": name, "candidate_id": cid,
                    "finding": "HISTORICAL_ACTION_BINDING_MISMATCH",
                })
        reference_groups[name] = {
            "sample_count": len(records),
            "reference_label_counts": dict(labels),
            "reference_by_original_policy_action": [
                {"reference": label, "action": action, "count": count}
                for (label, action), count in sorted(pairs.items())
            ],
            "independently_verified": group.get("independently_verified"),
            "historical_metrics": group.get("metrics"),
            "historical_interpretation": group.get("interpretation"),
            "promoted_to_official_no_answer_metric": False,
        }

    report = {
        "schema_version": "nb21c_action26_existing_evidence_v1",
        "status": (
            "ACTION26_INPUT_REPAIR_REQUIRED"
            if input_errors or binding_findings
            else "ACTION26_EXISTING_EVIDENCE_CONSUMED_REFERENCE_GATE_OPEN"
        ),
        "action": 26,
        "input_bindings": bindings,
        "input_errors": input_errors,
        "action_binding_findings": binding_findings,
        "historical_test_policy": {
            "population": len(index),
            "provenance": "CHATGPT_RECOVERY_STORED_POLICY",
            "original_action_counts": dict(route_counts),
            "review_mapped_to_no_answer": False,
            "clarification_reported_separately": True,
            "live_qwen_behavior_measured": False,
        },
        "historical_reference_groups": reference_groups,
        "recorded_verified_independent_reference_count":
            reference_report.get("verified_independent_reference_count"),
        "historical_reference_limitations":
            reference_report.get("limitations"),
        "C168_saved_result": summary(
            loaded.get("original_evidence_performance")
        ),
        "C197_saved_result": summary(
            loaded.get("model_reference_proposals")
        ),
        "official_no_answer": {
            "confusion_matrix": None,
            "precision": None, "recall": None, "f1": None,
            "reason": (
                "Accepted reference qualification and frozen no-answer "
                "action mapping have not been established by this package. "
                "Historical model proposals and delivery agreement are "
                "preserved without promotion to population ground truth."
            ),
        },
        "gate": {
            "status": "BLOCKED_NOT_PASS",
            "required": [
                "Qualified answerability reference for the declared population",
                "Accepted action mapping separating clarification and review",
                "Version/provenance bindings for any scored predictions",
            ],
            "development_population_assessed_by_this_cell": False,
            "project_wide_absence_of_reference_established": False,
        },
        "action_26_eligible_for_checkmark": False,
        "actions_22_25_reopened": False,
        "new_api_calls": 0,
        "new_semantic_judgments": 0,
        "historical_files_modified": False,
    }

    raw = json.dumps(
        report, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")
    file_hash = hashlib.sha256(raw).hexdigest()
    directory = root / (
        "artifacts/phase2/evaluation_and_ablation/work/outputs/"
        "NB21C.C003.action26_existing_evidence"
    )
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f"C003.report.{file_hash}.json"
    fd, temporary = tempfile.mkstemp(dir=directory, prefix=".C003-")
    try:
        with os.fdopen(fd, "wb") as stream:
            stream.write(raw)
            stream.flush()
            os.fsync(stream.fileno())
        try:
            os.link(temporary, target)
        except FileExistsError:
            require(target.read_bytes() == raw, "IMMUTABLE_OUTPUT_CONFLICT")
    finally:
        Path(temporary).unlink(missing_ok=True)

    require(target.read_bytes() == raw, "RELOAD_MISMATCH")
    print(json.dumps({
        **report,
        "output_path": target.relative_to(root).as_posix(),
        "output_sha256_file_bytes": file_hash,
        "reload_validation": "PASS",
    }, ensure_ascii=False, indent=2))
    return report


NB21C_C003_REPORT = nb21c_c003_run()

{
  "schema_version": "nb21c_action26_existing_evidence_v1",
  "status": "ACTION26_EXISTING_EVIDENCE_CONSUMED_REFERENCE_GATE_OPEN",
  "action": 26,
  "input_bindings": {
    "historical_policy": {
      "path": "artifacts/phase2/answerability_safety_confidence/work/outputs/nb20_test_policy_records_chatgpt_recovery_v1.jsonl",
      "sha256_file_bytes": "7363fca9ce6aa30e1020fb49fab8782fb1c87c4d3a3691dd3684bcf419dbf1ac",
      "expected_sha256": "7363fca9ce6aa30e1020fb49fab8782fb1c87c4d3a3691dd3684bcf419dbf1ac",
      "hash_matches": true
    },
    "historical_reference_metrics": {
      "path": "artifacts/phase2/answerability_safety_confidence/work/outputs/nb20_existing_reference_metrics_v1.json",
      "sha256_file_bytes": "b7149e251a9402bb47d526c7ac2bc0673a2b294d2a599536b3c6b861d93697bf",
      "expected_sha256": "b7149e251a9402bb47d526c7ac2bc0673a2b294d2a599536b3c6b861d93697bf",
      "hash_matches": true
    },
    "original_evidence_performance": {
      "path": "artifacts/phase2/a

<div dir="rtl" align="right">

## NB21C.C004 — اقدام ۲۶: جمع‌بندی نهایی شواهد و وضعیت Gate

گزارش C003 و مقایسهٔ موجود C198 مصرف می‌شوند.
ارجاعات C181، C189 و C197 با هش بایت فایل کنترل می‌شوند.

خروجی:
- جدول رفتار مشاهده‌شده در برابر پیشنهاد مرجع مدل؛
- تفکیک روشن‌سازی و نتیجهٔ نامعلوم؛
- حفظ محدودیت مرجع و منشأ تاریخی؛
- گزارش نهایی این دور و مانع سنجش رسمی No-answer.

این مرحله داوری یا آزمون مدل تازه ندارد.
عدد توصیفی به صحت قطعی یا F1 رسمی تبدیل نمی‌شود.
اقدام ۲۶ در وضعیت BLOCKED می‌ماند؛ تیک خودکار نمی‌خورد.
پس از ثبت موفق، ادامهٔ مستقل اقدام ۲۷ است.

</div>

In [6]:
# NB21C.C004 — Finalize the current action-26 evidence report.
# No API, historical execution, new judgments or checklist mutation.
def nb21c_c004_run():
    import os
    import json
    import hashlib
    import tempfile
    import subprocess
    from pathlib import Path
    from collections import Counter

    def need(ok, reason):
        if not ok:
            raise RuntimeError(reason)

    root = Path(subprocess.run(
        ["git", "--no-optional-locks", "rev-parse", "--show-toplevel"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()).resolve()

    def enc(value):
        return json.dumps(
            value, ensure_ascii=False, sort_keys=True,
            separators=(",", ":"), allow_nan=False,
        ).encode("utf-8")

    def sha(raw):
        return hashlib.sha256(raw).hexdigest()

    def path_for(relative):
        relative = Path(relative)
        need(not relative.is_absolute() and ".." not in relative.parts,
             "UNSAFE_RELATIVE_PATH")
        path = (root / relative).resolve()
        path.relative_to(root)
        return path

    bindings = {}

    def load(role, relative, expected):
        path = path_for(relative)
        raw = path.read_bytes()
        need(sha(raw) == expected, "HASH_MISMATCH:" + role)
        doc = json.loads(raw)
        need(isinstance(doc, dict), "EXPECTED_OBJECT:" + role)
        bindings[role] = {
            "path": relative, "sha256_file_bytes": expected,
        }
        return doc

    evaluation = "artifacts/phase2/evaluation_and_ablation/work/outputs/"
    upstream = "artifacts/phase2/answerability_safety_confidence/work/outputs/"

    h003 = "489ca642367c3bfcf12ced8efce5b304c842bc9f6191ca0c67cf3f622c89af6e"
    prior = load(
        "C003",
        evaluation + "NB21C.C003.action26_existing_evidence/"
        f"C003.report.{h003}.json",
        h003,
    )
    need(not prior["input_errors"], "C003_INPUT_ERRORS")
    need(not prior["action_binding_findings"], "C003_BINDING_FINDINGS")

    specs = {
        "C181": (
            "nb20c_historical_abstention_C181_v1/C181.inventory.",
            "9e5988d4cc7e50e8862dd81929ee09f84693ead0a6c97232581943de78c100aa",
        ),
        "C189": (
            "nb20c_behavior_consolidation_C189_v1/C189.report.",
            "a31fd52a5583fabe13fd4ab756adb0119280b37b71bce3379ce6ce1d1f8639b3",
        ),
        "C197": (
            "nb20c_model_reference_registry_C197_v1/C197.references.",
            "1a33dbdd0cb352794f7f3fa6228f0279f4a1d832cb573bea9e802211d2dcfc12",
        ),
        "C198": (
            "nb20c_abstention_comparison_C198_v1/C198.comparison.",
            "c0fbe625c12abfda2f57d7441fe2d575c2fd0457a1dd2061c909297bc9562b6a",
        ),
    }
    docs = {
        role: load(role, upstream + prefix + digest + ".json", digest)
        for role, (prefix, digest) in specs.items()
    }

    def index(rows, role):
        need(isinstance(rows, list), "EXPECTED_ROWS:" + role)
        result = {}
        for row in rows:
            cid = row.get("candidate_id")
            need(isinstance(cid, str) and cid and cid not in result,
                 "INVALID_OR_DUPLICATE_ID:" + role)
            result[cid] = row
        return result

    originals = index(docs["C181"]["cases"], "C181")
    observed = index(docs["C189"]["case_observations"], "C189")
    unknown = index(docs["C189"]["unknown_outcome_cases"], "C189_UNKNOWN")
    references = index(docs["C197"]["reference_proposals"], "C197")
    comparison = docs["C198"]

    need(len(originals) == 45 and len(observed) == 44 and len(unknown) == 1,
         "HISTORICAL_POPULATION_MISMATCH")
    need(not (set(observed) & set(unknown)), "OBSERVED_UNKNOWN_OVERLAP")
    need(set(observed) | set(unknown) == set(originals) == set(references),
         "POPULATION_JOIN_MISMATCH")

    for cid, original in originals.items():
        question = original["question"]
        need(isinstance(question, str), "INVALID_QUESTION")
        need(
            sha(question.encode("utf-8"))
            == original["query_sha256"]
            == references[cid]["query_sha256"],
            "QUESTION_VERSION_MISMATCH",
        )

    rows = comparison["case_comparisons"]
    saved = index(rows, "C198")
    need(set(saved) == set(observed), "COMPARISON_POPULATION_MISMATCH")
    matrix = {}
    for cid, observation in observed.items():
        behavior = observation["response_behavior"]
        reference = references[cid]["reference_behavior"]
        need(saved[cid]["observed_behavior"] == behavior,
             "SAVED_BEHAVIOR_MISMATCH")
        need(saved[cid]["proposed_reference_behavior"] == reference,
             "SAVED_REFERENCE_MISMATCH")
        matrix.setdefault(behavior, Counter())[reference] += 1
    matrix = {key: dict(value) for key, value in matrix.items()}
    need(matrix == comparison["observed_by_proposed_reference"],
         "MATRIX_RECONCILIATION_FAILED")
    need(comparison["qualified_independent_reference_established"] is False,
         "REFERENCE_QUALIFICATION_CHANGED_REVIEW_REQUIRED")

    report = {
        "schema_version": "nb21c_action26_round_summary_v1",
        "status": "ACTION26_CURRENT_ROUND_REPORTED_GATE_BLOCKED",
        "action": 26,
        "input_bindings": bindings,
        "validation": "HASH_POPULATION_QUESTION_AND_MATRIX_BINDINGS_PASS",
        "historical_policy": prior["historical_test_policy"],
        "limited_reference_groups": prior["historical_reference_groups"],
        "C168_acceptance_reasons":
            prior["C168_saved_result"]["acceptance_reasons"],
        "behavior_comparison": {
            "scope": comparison["completion_scope"],
            "target_count": 45,
            "observed_count": 44,
            "unknown_outcome_count": 1,
            "unknown_candidate_ids": sorted(unknown),
            "observed_behavior_counts":
                comparison["observed_behavior_counts"],
            "observed_by_model_proposed_reference": matrix,
            "reference_behavior_counts_all_45":
                comparison["proposed_reference_behavior_counts_all_45"],
            "strict_model_reference_abstention_support":
                comparison["strict_model_reference_abstention_support"],
            "verified_semantic_error_count": None,
            "population_accuracy": None,
        },
        "official_no_answer": {
            "confusion_matrix": None,
            "precision": None, "recall": None, "f1": None,
            "status": "NOT_ESTIMABLE_UNDER_ESTABLISHED_REFERENCE_CONTRACT",
            "blockers": [
                "Qualified semantic answerability reference not established",
                "Clarification cannot be relabeled as unanswerable",
                "REVIEW cannot be relabeled as executed no-answer",
                "Historical targeted sample cannot establish population performance",
            ],
        },
        "limitations": [
            "Comparison uses provisional model reference proposals.",
            "37 observed abstentions are not 37 verified policy errors.",
            "0/37 model-reference support is not verified abstention precision.",
            "Unknown outcome is preserved and is not a model error.",
            "Historical Test evidence is not an untouched new evaluation.",
            "No project-wide absence of usable reference is established.",
            "Development population performance is not measured here.",
        ],
        "gate_status": "BLOCKED_NOT_PASS",
        "current_evidence_round_finished": True,
        "action_26_completed": False,
        "action_26_eligible_for_checkmark": False,
        "checklist_updated": False,
        "actions_22_25_reopened": False,
        "new_api_calls": 0,
        "new_semantic_judgments": 0,
        "next_action": 27,
        "next_action_scope": "CONSUME_EXISTING_SAFETY_EVIDENCE",
    }

    body = enc(report)
    digest = sha(body)
    directory = path_for(
        evaluation + "NB21C.C004.action26_round_summary"
    )
    directory.mkdir(parents=True, exist_ok=True)

    def save_immutable(path, raw):
        fd, temporary = tempfile.mkstemp(dir=directory, prefix=".C004-")
        try:
            with os.fdopen(fd, "wb") as stream:
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())
            try:
                os.link(temporary, path)
            except FileExistsError:
                need(path.read_bytes() == raw, "IMMUTABLE_CONFLICT")
        finally:
            Path(temporary).unlink(missing_ok=True)
        need(path.read_bytes() == raw, "RELOAD_FAILED")

    json_path = directory / f"C004.report.{digest}.json"
    save_immutable(json_path, body)

    text = """# NB21 — اقدام ۲۶: جمع‌بندی این دور

وضعیت اقدام: BLOCKED؛ تیک نخورده است.
جمع‌بندی شواهد موجود در این دور پایان یافت؛ ادامهٔ مستقل: اقدام ۲۷.

## نتایج قابل‌گزارش

- Policy تاریخی ۱۱۴۹ پرونده: ۱۰۵۶ ANSWER، ۸۹ REVIEW و چهار CLARIFY.
- این Policy منشأ ChatGPT recovery دارد؛ رفتار زندهٔ Qwen نیست.
- بستهٔ هدفمند ۴۵پرونده‌ای: ۴۴ رفتار مشاهده‌شده و یک نتیجهٔ نامعلوم.
- رفتار مشاهده‌شده: ۳۷ خودداری، دو روشن‌سازی و پنج پاسخ محتوایی.
- پیشنهاد مرجع مدل برای تمام ۴۵ مورد: ۳۳ پاسخ کامل، پنج پاسخ محدود و هفت روشن‌سازی.
- از ۳۷ خودداری، پیشنهاد مدل برای ۳۱ مورد پاسخ کامل/محدود و برای شش مورد روشن‌سازی بوده است.

این اختلاف‌ها خطای قطعی اثبات‌شده نیستند.
نسبت تاریخی ۰/۳۷، پشتیبانی پیشنهاد مرجع مدل از خودداری است؛
Precision معتبر خودداری محسوب نمی‌شود.

## مانع Gate

مرجع معنایی پذیرفته‌شده و قرارداد معتبر نگاشت خودداری
در این بسته اثبات نشده‌اند؛ Precision، Recall و F1 رسمی نامعین‌اند.
روشن‌سازی و REVIEW به غیرقابل‌پاسخ یا خودداری صحیح تبدیل نشده‌اند.
نتایج نمونه به جمعیت تعمیم داده نشده‌اند.
نبود مرجع در تمام پروژه نیز ادعا نمی‌شود.

## ادامه

بدون API یا نجات تک‌پرونده‌ای، به اقدام ۲۷ می‌رویم.
رفع مانع اقدام ۲۶ برای Gate وابسته همچنان لازم است.
اقدامات ۲۲ تا ۲۵ بسته می‌مانند.
"""
    text += f"\nگزارش داده: `{json_path.relative_to(root).as_posix()}`\n"
    text += f"\nSHA256 بایت گزارش داده: `{digest}`\n"
    md_raw = text.encode("utf-8")
    md_path = directory / f"C004.summary.{sha(md_raw)}.md"
    save_immutable(md_path, md_raw)

    print(json.dumps({
        "status": report["status"],
        "validation": report["validation"],
        "observed_behavior_counts":
            report["behavior_comparison"]["observed_behavior_counts"],
        "unknown_outcome_count": 1,
        "official_no_answer": report["official_no_answer"],
        "current_evidence_round_finished": True,
        "action_26_completed": False,
        "report_path": json_path.relative_to(root).as_posix(),
        "report_sha256_file_bytes": digest,
        "summary_path": md_path.relative_to(root).as_posix(),
        "reload_validation": "PASS",
        "new_api_calls": 0,
        "next_action": 27,
    }, ensure_ascii=False, indent=2))
    return report


NB21C_C004_REPORT = nb21c_c004_run()

{
  "status": "ACTION26_CURRENT_ROUND_REPORTED_GATE_BLOCKED",
  "validation": "HASH_POPULATION_QUESTION_AND_MATRIX_BINDINGS_PASS",
  "observed_behavior_counts": {
    "ABSTENTION_ONLY": 37,
    "CLARIFICATION_ONLY": 2,
    "SUBSTANTIVE_RESPONSE": 5
  },
  "unknown_outcome_count": 1,
  "official_no_answer": {
    "confusion_matrix": null,
    "precision": null,
    "recall": null,
    "f1": null,
    "status": "NOT_ESTIMABLE_UNDER_ESTABLISHED_REFERENCE_CONTRACT",
    "blockers": [
      "Qualified semantic answerability reference not established",
      "Clarification cannot be relabeled as unanswerable",
      "REVIEW cannot be relabeled as executed no-answer",
      "Historical targeted sample cannot establish population performance"
    ]
  },
  "current_evidence_round_finished": true,
  "action_26_completed": false,
  "report_path": "artifacts/phase2/evaluation_and_ablation/work/outputs/NB21C.C004.action26_round_summary/C004.report.225876dd4c8ffe32bf99d1ab2978ab4507082db1b8d56009f7b

<div dir="rtl" align="right">

## NB21C.C005 — اقدام ۲۶: بررسی تجمیعی قرارداد، مرجع و شواهد بازاستفاده

### هدف

مشخص‌کردن اینکه برای سنجش پاسخ/خودداری چه قراردادها،
برچسب‌ها و تصمیم‌های قبلی واقعاً موجودند و چه محدودیتی دارند.

### بررسی‌ها

- خواندن ایستای NB20، NB20b، NB20c، NB21 و NB21b.
- استخراج ارجاعات مرتبط از کد و خروجی ذخیره‌شده، بدون اجرای آن‌ها.
- بررسی فایل‌های مرتبط در پوشهٔ خروجی NB20 و NB21.
- استخراج وضعیت پذیرش، تعریف رفتارها، برچسب‌ها، منشأ،
  استقلال، تعداد و محدودیت‌های مرجع.
- تفکیک نتیجهٔ تاریخی، پیشنهاد مدل و مرجع پذیرفته‌شده.
- حفظ فایل‌های مفقود، خطاهای خواندن و تعارض هش در گزارش.

### خروجی

گزارش جامع و قابل بازخوانی برای تعیین مسیر اقدام ۲۶.
وجود فیلد یا عبارت پذیرش، به‌تنهایی تأیید معنایی نیست.
فقدان فایل در محدودهٔ بررسی، فقدان آن در کل پروژه محسوب نمی‌شود.

هیچ API، بازداوری، اجرای سل تاریخی یا تغییر چک‌لیست انجام نمی‌شود.
فقط گزارش جدید و نسخه‌دار نوشته می‌شود.

</div>

In [7]:
# NB21C.C005 — Bounded, aggregate action-26 evidence inspection.
# Reads data/source only. Never executes historical code or calls an API.
def nb21c_c005_run():
    import ast
    import os
    import re
    import json
    import hashlib
    import tempfile
    import subprocess
    from pathlib import Path
    from collections import Counter, defaultdict

    root = Path(subprocess.run(
        ["git", "--no-optional-locks", "rev-parse", "--show-toplevel"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()).resolve()

    def sha(raw):
        return hashlib.sha256(raw).hexdigest()

    def encode(value):
        return json.dumps(
            value, ensure_ascii=False, sort_keys=True,
            separators=(",", ":"), allow_nan=False,
        ).encode("utf-8")

    def need(ok, reason):
        if not ok:
            raise RuntimeError(reason)

    def safe_path(relative):
        relative = Path(relative)
        need(not relative.is_absolute() and ".." not in relative.parts,
             "UNSAFE_PATH")
        path = (root / relative).resolve()
        path.relative_to(root)
        return path

    def strict_json(raw):
        def pairs(items):
            result = {}
            for key, value in items:
                need(key not in result, "DUPLICATE_JSON_KEY")
                result[key] = value
            return result

        def reject(value):
            raise ValueError("NONSTANDARD_JSON_CONSTANT:" + value)

        return json.loads(
            raw, object_pairs_hook=pairs, parse_constant=reject,
        )

    notebooks = [
        "20_answerability_safety_confidence.ipynb",
        "20b_answerability_safety_confidence_scoped_review.ipynb",
        "20c_answerability_safety_confidence_audit_and_acceptance.ipynb",
        "21_evaluation_and_ablation.ipynb",
        "21b_evaluation_and_ablation_continued.ipynb",
    ]
    scan_roots = [
        "artifacts/phase2/answerability_safety_confidence/work/outputs",
        "artifacts/phase2/evaluation_and_ablation/work/outputs",
    ]
    relevant = re.compile(
        r"answerability|abstention|no.?answer|reference|sufficiency|"
        r"behavior|clarification|gold_comparison|adoption|"
        r"پاسخ.?پذیر|خودداری|مرجع",
        re.I,
    )
    decisive = re.compile(
        r"qualif|accept|independen|verified_ground|ground_truth|"
        r"reference_status|reference_method|reference_behavior|"
        r"reference_answerability|mapping|denominator|"
        r"completion_scope|metric_scope|limitation|blocker|"
        r"formal_acceptance|action_.*completed|full_policy",
        re.I,
    )
    field_of_interest = re.compile(
        r"reference|answerability|sufficiency|response_behavior|"
        r"final_action|verified_ground_truth|independen|"
        r"acceptance|qualification|clarification",
        re.I,
    )

    references = defaultdict(list)
    source_records, notebook_errors = [], []
    notebook_bindings = []
    candidate_paths = set()

    def extract_paths(value, origin, depth=0):
        if depth > 40:
            return
        if isinstance(value, dict):
            for key, item in value.items():
                if isinstance(key, str):
                    extract_paths(key, origin, depth + 1)
                extract_paths(item, origin, depth + 1)
        elif isinstance(value, list):
            for item in value:
                extract_paths(item, origin, depth + 1)
        elif isinstance(value, str):
            # Only literal project-relative file paths; no eval/format/runpy.
            for match in re.finditer(
                r"(?:artifacts|reports|data)/[^\s\"'<>`]+?\.jsonl?",
                value,
            ):
                relative = match.group(0)
                if any(token in relative for token in ("{", "}", "\\", "..")):
                    continue
                if relevant.search(relative):
                    candidate_paths.add(relative)
                    if origin not in references[relative]:
                        references[relative].append(origin)

    for name in notebooks:
        relative = "notebooks/phase2/" + name
        path = safe_path(relative)
        if not path.is_file():
            notebook_errors.append({"path": relative, "error": "MISSING"})
            continue
        try:
            raw = path.read_bytes()
            doc = strict_json(raw)
            notebook_bindings.append({
                "path": relative, "sha256_file_bytes": sha(raw),
                "cell_count": len(doc["cells"]),
            })
            for number, cell in enumerate(doc["cells"], 1):
                source = cell.get("source", [])
                source = "".join(source) if isinstance(source, list) else source
                if not isinstance(source, str):
                    continue
                origin = f"{name}:saved_cell_{number}"
                if relevant.search(source):
                    lines = source.splitlines()
                    source_records.append({
                        "origin": origin,
                        "cell_type": cell.get("cell_type"),
                        "header": lines[0] if lines else "",
                        "relevant_lines": [
                            {"line": i, "text": line}
                            for i, line in enumerate(lines, 1)
                            if relevant.search(line) or decisive.search(line)
                        ],
                    })
                if cell.get("cell_type") == "code":
                    try:
                        tree = ast.parse(source)
                        for node in ast.walk(tree):
                            if isinstance(node, ast.Constant):
                                if isinstance(node.value, str):
                                    extract_paths(node.value, origin)
                    except SyntaxError:
                        # Notebook magics may not parse; inspect literal text only.
                        extract_paths(source, origin)
                else:
                    extract_paths(source, origin)

                # Inspect saved output text as data; never execute it.
                for output in cell.get("outputs", []):
                    text = output.get("text", [])
                    text = "".join(text) if isinstance(text, list) else text
                    if isinstance(text, str):
                        extract_paths(text, origin + ":output")
                    data = output.get("data", {})
                    if isinstance(data, dict):
                        extract_paths(data.get("application/json"),
                                      origin + ":json_output")
        except Exception as error:
            notebook_errors.append({
                "path": relative, "error_type": type(error).__name__,
            })

    # Restrict discovery to the two established evidence directories.
    # Raw provider bodies and caches are not included merely by their location.
    for relative in scan_roots:
        directory = safe_path(relative)
        if not directory.is_dir():
            notebook_errors.append({"path": relative, "error": "MISSING_DIRECTORY"})
            continue
        for path in directory.rglob("*"):
            if path.is_symlink() or not path.is_file():
                continue
            if path.suffix.lower() not in (".json", ".jsonl"):
                continue
            if not relevant.search(path.name):
                continue
            if path.name.lower() in ("raw.json", "response.json"):
                continue
            # Exclude reports created by this ongoing discovery.
            if "NB21C.C005" in str(path):
                continue
            candidate_paths.add(path.relative_to(root).as_posix())

    inspected, unread, field_profiles = [], [], []
    max_bytes = 40 * 1024 * 1024

    def inspect_document(doc):
        facts, profiles, row_lists = [], defaultdict(Counter), {}
        id_count = 0

        def walk(value, pointer="", depth=0):
            nonlocal id_count
            if depth > 40:
                return
            if isinstance(value, dict):
                if isinstance(value.get("candidate_id"), str):
                    id_count += 1
                for key, item in value.items():
                    location = pointer + "/" + str(key)
                    if isinstance(item, (dict, list)):
                        walk(item, location, depth + 1)
                    else:
                        if decisive.search(str(key)):
                            facts.append({"pointer": location, "value": item})
                        if field_of_interest.search(str(key)):
                            # Aggregate machine labels and flags, not prose.
                            if isinstance(item, bool) or item is None:
                                profiles[location][str(item)] += 1
                            elif (
                                isinstance(item, str) and len(item) <= 140
                                and re.fullmatch(r"[A-Za-z0-9_.:/ -]+", item)
                            ):
                                profiles[location][item] += 1
            elif isinstance(value, list):
                row_lists[pointer or "/"] = len(value)
                for item in value:
                    walk(item, pointer + "/*", depth + 1)

        walk(doc)
        return {
            "top_level_fields": sorted(doc) if isinstance(doc, dict) else None,
            "root_type": type(doc).__name__,
            "root_list_count": len(doc) if isinstance(doc, list) else None,
            "candidate_id_occurrences_not_unique_cases": id_count,
            "row_list_lengths": row_lists,
            "qualification_and_contract_facts": facts,
            "label_profiles": {
                key: dict(value) for key, value in profiles.items()
            },
        }

    for relative in sorted(candidate_paths):
        try:
            path = safe_path(relative)
            if not path.is_file():
                unread.append({"path": relative, "reason": "MISSING"})
                continue
            if path.stat().st_size > max_bytes:
                unread.append({
                    "path": relative, "reason": "SIZE_LIMIT",
                    "bytes": path.stat().st_size,
                })
                continue
            raw = path.read_bytes()
            digest = sha(raw)
            # Check only an explicit full SHA256 suffix, not canonical/text hashes.
            suffix = re.search(r"\.([0-9a-f]{64})\.jsonl?$", path.name)
            suffix_match = (
                digest == suffix.group(1) if suffix else None
            )
            doc = (
                [strict_json(line) for line in raw.splitlines() if line.strip()]
                if path.suffix == ".jsonl" else strict_json(raw)
            )
            item = {
                "path": relative, "bytes": len(raw),
                "sha256_file_bytes": digest,
                "filename_digest_matches_file_bytes": suffix_match,
                "filename_hash_semantics_not_assumed": True,
                "referenced_by": references.get(relative, []),
                **inspect_document(doc),
            }
            inspected.append(item)
            field_profiles.append({
                "path": relative,
                "label_profiles": item["label_profiles"],
            })
        except Exception as error:
            unread.append({
                "path": relative, "reason": "READ_OR_SCHEMA_ERROR",
                "error_type": type(error).__name__,
            })

    # Prioritize explicit qualification statements, not generic status=PASS.
    candidates = []
    for item in inspected:
        statements = [
            fact for fact in item["qualification_and_contract_facts"]
            if re.search(
                r"qualified|independent_reference|verified_ground_truth|"
                r"reference_qualification|reference_semantic_correctness",
                fact["pointer"], re.I,
            )
            and fact["value"] is True
        ]
        if statements:
            candidates.append({
                "path": item["path"],
                "positive_qualification_statements": statements,
                "interpretation": "CLAIM_REQUIRES_SCOPE_REVIEW_NOT_AUTO_ACCEPTANCE",
            })

    report = {
        "schema_version": "nb21c_action26_aggregate_reference_inspection_v1",
        "status": "AGGREGATE_INSPECTION_COMPLETE_REQUIRES_EVIDENCE_INTERPRETATION",
        "action": 26,
        "scope": {
            "notebooks": notebook_bindings,
            "artifact_scan_roots": scan_roots,
            "file_selection": "RELEVANT_LITERAL_REFERENCES_AND_CONTROL_FILENAMES",
            "maximum_file_bytes": max_bytes,
            "historical_source_executed": False,
            "all_project_files_read": False,
        },
        "notebook_errors": notebook_errors,
        "source_contract_and_decision_records": source_records,
        "inspected_artifacts": inspected,
        "unread_or_missing": unread,
        "positive_qualification_claim_candidates": candidates,
        "automatic_reference_acceptance": False,
        "official_no_answer_metrics_computed": False,
        "action_26_completed": False,
        "project_wide_reference_absence_established": False,
        "new_api_calls": 0,
        "new_semantic_judgments": 0,
        "checklist_modified": False,
    }

    raw = encode(report)
    digest = sha(raw)
    directory = safe_path(
        "artifacts/phase2/evaluation_and_ablation/work/outputs/"
        "NB21C.C005.aggregate_reference_inspection"
    )
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f"C005.report.{digest}.json"
    fd, temporary = tempfile.mkstemp(dir=directory, prefix=".C005-")
    try:
        with os.fdopen(fd, "wb") as stream:
            stream.write(raw)
            stream.flush()
            os.fsync(stream.fileno())
        try:
            os.link(temporary, target)
        except FileExistsError:
            need(target.read_bytes() == raw, "IMMUTABLE_REPORT_CONFLICT")
    finally:
        Path(temporary).unlink(missing_ok=True)
    need(target.read_bytes() == raw, "REPORT_RELOAD_FAILED")

    # Print compact per-artifact results; full detail remains in the report.
    summaries = []
    for item in inspected:
        top_facts = [
            fact for fact in item["qualification_and_contract_facts"]
            if fact["pointer"].count("/") <= 2
        ]
        labels = {
            key: value for key, value in item["label_profiles"].items()
            if re.search(
                r"reference_behavior|reference_answerability|"
                r"response_behavior|final_action|verified_ground_truth",
                key, re.I,
            )
        }
        if top_facts or labels:
            summaries.append({
                "path": item["path"],
                "facts": top_facts,
                "label_counts": labels,
            })

    print(json.dumps({
        "status": report["status"],
        "notebooks_read": notebook_bindings,
        "source_records_count": len(source_records),
        "artifact_count": len(inspected),
        "notebook_errors": notebook_errors,
        "unread_or_missing": unread,
        "qualification_claim_candidates": candidates,
        "artifact_summaries": summaries,
        "report_path": target.relative_to(root).as_posix(),
        "report_sha256_file_bytes": digest,
        "reload_validation": "PASS",
        "new_api_calls": 0,
        "action_26_completed": False,
    }, ensure_ascii=False, indent=2))
    return report


NB21C_C005_REPORT = nb21c_c005_run()

{
  "status": "AGGREGATE_INSPECTION_COMPLETE_REQUIRES_EVIDENCE_INTERPRETATION",
  "notebooks_read": [
    {
      "path": "notebooks/phase2/20_answerability_safety_confidence.ipynb",
      "sha256_file_bytes": "6d1d03990c8778c1d0c6b3b3cb8c18d2c6ce9062aa4888c062283b75656302f7",
      "cell_count": 700
    },
    {
      "path": "notebooks/phase2/20b_answerability_safety_confidence_scoped_review.ipynb",
      "sha256_file_bytes": "d7232ac890ab75e0430a768e0bd433bee8ce4843fc980214d15bac16eb88af8b",
      "cell_count": 766
    },
    {
      "path": "notebooks/phase2/20c_answerability_safety_confidence_audit_and_acceptance.ipynb",
      "sha256_file_bytes": "2ea60eaa26fa1d1750459f4812d074469e8e8384be407141f649fd4c8ddc0634",
      "cell_count": 906
    },
    {
      "path": "notebooks/phase2/21_evaluation_and_ablation.ipynb",
      "sha256_file_bytes": "1a0a006922ca8698af77b29f959a5c0eb3edb76881c8f2627a9b258da9352a94",
      "cell_count": 694
    },
    {
      "path": "notebooks/phase2/21b

<div dir="rtl" align="right">

## NB21C.C006 — تجمیع آفلاین سوابق ۱۱۴۹ پروندهٔ Test

### هدف

اتصال پرونده‌های Test به ورودی، پاسخ ثبت‌شده، بررسی تاریخی،
شواهد پشتیبان، تصمیم Policy و داوری‌های ذخیره‌شده؛
برای مشخص‌کردن پوشش کارهای قبلی و جلوگیری از داوری تکراری.

### بررسی‌ها

- تثبیت جمعیت ۱۱۴۹تایی با بستهٔ فنی تاریخی و کنترل هش منابع اصلی.
- تطبیق شناسه‌ها میان ورودی، پاسخ، Audit، Support، Policy و Delivery.
- بررسی تطابق هش پاسخ با سوابق Audit و Support.
- جست‌وجوی فایل‌های JSON و JSONL در دو پوشهٔ خروجی مشخص NB20 و NB21.
- استخراج اطلاعات ثبت‌شده دربارهٔ روش داوری، پاسخ‌پذیری،
  کفایت شواهد، خودداری، روشن‌سازی، ایمنی و وضعیت پذیرش.
- ثبت فایل‌های خوانده‌نشده، خطاهای خواندن و اتصال‌های ناقص.

### حدود تفسیر

- تعداد فایل‌ها یا ارجاعات، تعداد داوری‌های مستقل نیست.
- اتصال شناسه به‌تنهایی تطابق نسخهٔ سؤال، شواهد و پاسخ را ثابت نمی‌کند.
- وجود داوری یا عبارت پذیرش، خودکار به معنی مرجع معتبر اقدام ۲۶ نیست.
- نبود ارجاع متصل، ثابت نمی‌کند پرونده قبلاً داوری نشده است.
- بررسی به پوشه‌های اعلام‌شده محدود است؛ ممیزی تمام پروژه نیست.
- فایل‌های خام مشخص‌شده و فایل‌های بزرگ‌تر از ۶۴ مگابایت خوانده نمی‌شوند.
- نتایج تاریخی، ارزیابی زندهٔ Qwen محسوب نمی‌شوند.

### خروجی

گزارش نسخه‌دار شامل سوابق هر پرونده، فهرست فایل‌های بررسی‌شده،
مشکلات اتصال و محدودیت‌های پوشش.

پس از تفسیر گزارش مشخص می‌شود کدام داوری‌ها قابل‌بازاستفاده‌اند
و آیا داوری تازه‌ای لازم است.

هیچ API، اجرای کد تاریخی، داوری معنایی تازه، محاسبهٔ معیار رسمی
یا تغییر چک‌لیست انجام نمی‌شود. اقدام ۲۶ خودکار تأیید نمی‌شود.

</div>

In [8]:
# NB21C.C006 — Aggregate offline inventory of all 1149 historical Test cases.
# No API, historical-code execution, new judgment, or checklist modification.

def nb21c_c006_run():
    import os
    import re
    import json
    import hashlib
    import tempfile
    import subprocess
    from pathlib import Path
    from collections import Counter, defaultdict

    def need(condition, message):
        if not condition:
            raise RuntimeError(message)

    def sha(raw):
        return hashlib.sha256(raw).hexdigest()

    def encode(value):
        return json.dumps(
            value, ensure_ascii=False, sort_keys=True,
            separators=(",", ":"), allow_nan=False,
        ).encode("utf-8")

    def strict_json(raw):
        def pairs(items):
            result = {}
            for key, value in items:
                need(key not in result, "DUPLICATE_JSON_KEY")
                result[key] = value
            return result

        def reject(value):
            raise ValueError("NONSTANDARD_JSON_CONSTANT:" + value)

        return json.loads(
            raw, object_pairs_hook=pairs, parse_constant=reject,
        )

    root = Path(subprocess.run(
        ["git", "--no-optional-locks", "rev-parse", "--show-toplevel"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()).resolve()

    def safe_path(relative):
        relative = Path(relative)
        need(
            not relative.is_absolute() and ".." not in relative.parts,
            "UNSAFE_RELATIVE_PATH",
        )
        current = root
        for part in relative.parts:
            current = current / part
            need(not current.is_symlink(), "SYMLINK_NOT_ALLOWED")
        path = current.resolve()
        path.relative_to(root)
        return path

    output_roots = [
        "artifacts/phase2/answerability_safety_confidence/work/outputs",
        "artifacts/phase2/evaluation_and_ablation/work/outputs",
    ]
    package_relative = (
        output_roots[0] + "/nb20_final_technical_package_v1.json"
    )
    package_raw = safe_path(package_relative).read_bytes()
    need(
        sha(package_raw)
        == "d86290d281ba97351344db95124d3ae428d6c6525869471cc5b581b4501c5ec1",
        "TECHNICAL_PACKAGE_VERSION_CHANGED",
    )
    package = strict_json(package_raw)
    need(package.get("candidate_count") == 1149, "PACKAGE_COUNT_MISMATCH")

    # Anchor the population to the established historical package.
    required_roles = ("inputs", "active", "audit", "support", "policy", "delivery")
    role_indexes = {}
    bindings = {}
    core_problems = []

    def candidate_id(row):
        if not isinstance(row, dict):
            return None
        value = row.get("candidate_id")
        if isinstance(value, str):
            return value
        decision = row.get("decision")
        if isinstance(decision, dict):
            value = decision.get("candidate_id")
            if isinstance(value, str):
                return value
        return None

    for role in required_roles:
        spec = package["source_files"][role]
        relative = spec["relative_path"]
        raw = safe_path(relative).read_bytes()
        need(sha(raw) == spec["sha256"], "CORE_HASH_MISMATCH:" + role)
        if "bytes" in spec:
            need(len(raw) == spec["bytes"], "CORE_SIZE_MISMATCH:" + role)

        index = {}
        for line_number, line in enumerate(raw.splitlines(), 1):
            if not line.strip():
                continue
            row = strict_json(line)
            cid = candidate_id(row)
            need(cid is not None, f"CORE_ID_MISSING:{role}:{line_number}")
            need(cid not in index, f"CORE_DUPLICATE_ID:{role}:{cid}")
            index[cid] = row

        role_indexes[role] = index
        bindings[role] = {
            "path": relative,
            "sha256_file_bytes": sha(raw),
            "record_count": len(index),
        }

    population = set(role_indexes["policy"])
    need(len(population) == 1149, "POLICY_POPULATION_NOT_1149")
    for role, index in role_indexes.items():
        need(set(index) == population, "CORE_POPULATION_MISMATCH:" + role)

    for cid in sorted(population):
        audit = role_indexes["audit"][cid]
        support = role_indexes["support"][cid]
        active = role_indexes["active"][cid]
        policy = role_indexes["policy"][cid]

        if policy.get("split") != "test":
            core_problems.append({
                "candidate_id": cid, "problem": "POLICY_SPLIT_NOT_TEST",
            })

        response = active.get("response")
        response_hash = sha(encode(response))
        recorded_hashes = {
            "active": active.get("response_sha256"),
            "audit": audit.get("response_sha256"),
            "support": support.get("response_sha256"),
        }
        if any(value != response_hash for value in recorded_hashes.values()):
            core_problems.append({
                "candidate_id": cid,
                "problem": "ACTIVE_AUDIT_SUPPORT_RESPONSE_HASH_MISMATCH",
                "computed_canonical_response_sha256": response_hash,
                "recorded_hashes": recorded_hashes,
            })

    # Discover stored review/control artifacts only inside the stated roots.
    # These are recorded claims, not fresh semantic judgments.
    semantic_fields = re.compile(
        r"answerability|abstention|sufficien|reference_behavior|"
        r"response_behavior|semantic|factual|claim_support|"
        r"question_answer_fit|safety|critical|clarif|"
        r"acceptance|qualification|verified|independen|"
        r"review_method|reviewer|gold_used|context_independence|"
        r"final_action|recovery_status|verdict|assessment",
        re.I,
    )
    identity_fields = {
        "candidate_id", "case_id", "sample_rank", "historical_position",
        "query_sha256", "response_sha256", "active_response_sha256",
        "source_response_sha256", "reference_method", "method",
        "review_cohort", "reviewer_identity", "model", "status",
        "completion_scope", "metric_scope", "evaluation_view",
        "input_condition", "requirement_id", "checklist_action_number",
    }
    excluded_names = {
        "raw.json", "response.json", "request.json", "body.json",
    }
    max_bytes = 64 * 1024 * 1024
    max_depth = 60

    artifacts = []
    issues = []
    per_case = defaultdict(list)
    core_paths = {item["path"] for item in bindings.values()}
    case_mentions = Counter()
    files_with_case_evidence = set()

    def selected_fields(obj):
        result = {}
        for key, value in obj.items():
            if key in identity_fields or semantic_fields.search(str(key)):
                # Preserve scalar statements exactly; don't copy whole payloads.
                if value is None or isinstance(value, (str, bool, int, float)):
                    result[str(key)] = value
                elif isinstance(value, list) and all(
                    item is None or isinstance(item, (str, bool, int, float))
                    for item in value
                ):
                    result[str(key)] = value
        return result

    def inspect_document(doc, relative, location):
        global_facts = []
        hit_count = 0

        def walk(value, pointer, owner=None, depth=0):
            nonlocal hit_count
            if depth > max_depth:
                issues.append({
                    "path": relative, "location": pointer,
                    "reason": "NESTING_LIMIT",
                })
                return

            if isinstance(value, dict):
                explicit = candidate_id(value)
                # case_id may itself contain the full canonical candidate ID.
                if explicit is None and value.get("case_id") in population:
                    explicit = value["case_id"]

                if explicit is not None:
                    owner = explicit if explicit in population else None

                facts = selected_fields(value)
                if facts:
                    record = {"location": pointer, "recorded_fields": facts}
                    if owner in population:
                        per_case[owner].append({
                            "path": relative, **record,
                            "association": (
                                "EXPLICIT_ID" if explicit == owner
                                else "NESTED_UNDER_IDENTIFIED_RECORD"
                            ),
                        })
                        case_mentions[owner] += 1
                        hit_count += 1
                    elif depth <= 2:
                        global_facts.append(record)

                for key, item in value.items():
                    child_owner = key if key in population else owner
                    escaped = str(key).replace("~", "~0").replace("/", "~1")
                    walk(
                        item, pointer + "/" + escaped,
                        child_owner, depth + 1,
                    )

            elif isinstance(value, list):
                for number, item in enumerate(value):
                    walk(item, pointer + "/" + str(number), owner, depth + 1)

        walk(doc, location)
        return global_facts, hit_count

    candidates = set()
    for relative_root in output_roots:
        directory = safe_path(relative_root)
        if not directory.is_dir():
            issues.append({"path": relative_root, "reason": "MISSING_DIRECTORY"})
            continue

        for folder, directories, filenames in os.walk(directory, followlinks=False):
            directories[:] = sorted(
                name for name in directories
                if not (Path(folder) / name).is_symlink()
                and not name.startswith("NB21C.C006")
            )
            for name in sorted(filenames):
                path = Path(folder) / name
                if path.is_symlink():
                    continue
                if path.suffix.lower() not in {".json", ".jsonl"}:
                    continue
                if name.lower() in excluded_names:
                    continue
                candidates.add(path.relative_to(root).as_posix())

    print(
        f"C006: core population bound: {len(population)}; "
        f"stored artifacts queued: {len(candidates)}",
        flush=True,
    )

    for number, relative in enumerate(sorted(candidates), 1):
        if relative in core_paths:
            continue
        try:
            path = safe_path(relative)
            size = path.stat().st_size
            if size > max_bytes:
                issues.append({
                    "path": relative, "reason": "SIZE_LIMIT", "bytes": size,
                })
                continue

            raw = path.read_bytes()
            file_facts = []
            hits = 0
            parse_failures = 0

            if path.suffix.lower() == ".jsonl":
                for line_number, line in enumerate(raw.splitlines(), 1):
                    if not line.strip():
                        continue
                    try:
                        doc = strict_json(line)
                        facts, count = inspect_document(
                            doc, relative, f"line:{line_number}",
                        )
                        file_facts.extend(facts)
                        hits += count
                    except Exception as error:
                        parse_failures += 1
                        issues.append({
                            "path": relative, "line": line_number,
                            "reason": "JSONL_RECORD_ERROR",
                            "error_type": type(error).__name__,
                            "message": str(error)[:200],
                        })
            else:
                doc = strict_json(raw)
                file_facts, hits = inspect_document(doc, relative, "")

            artifacts.append({
                "path": relative,
                "bytes": len(raw),
                "sha256_file_bytes": sha(raw),
                "case_fact_occurrences_not_unique_reviews": hits,
                "file_level_recorded_facts": file_facts,
                "parse_failures": parse_failures,
            })
            if hits:
                files_with_case_evidence.add(relative)

        except Exception as error:
            issues.append({
                "path": relative, "reason": "READ_OR_PARSE_ERROR",
                "error_type": type(error).__name__,
                "message": str(error)[:200],
            })

        if number % 500 == 0:
            print(f"C006: scanned {number}/{len(candidates)}", flush=True)

    # Keep original historical judgments and later evidence separately.
    records = []
    for cid in sorted(population):
        audit = role_indexes["audit"][cid]
        support = role_indexes["support"][cid]
        policy = role_indexes["policy"][cid]
        delivery = role_indexes["delivery"][cid]
        inputs = role_indexes["inputs"][cid]
        active = role_indexes["active"][cid]

        records.append({
            "candidate_id": cid,
            "population": "HISTORICAL_TEST_1149",
            "core_roles_present": list(required_roles),
            "input_messages_sha256_recorded": inputs.get("messages_sha256"),
            "active_response_sha256_recorded": active.get("response_sha256"),
            "historical_audit": audit,
            "historical_support": support,
            "historical_policy_decision": policy.get("decision"),
            "historical_internal_delivery_action":
                delivery.get("internal_delivery_action"),
            "additional_artifact_facts": per_case.get(cid, []),
            "additional_artifact_count": len({
                item["path"] for item in per_case.get(cid, [])
            }),
            "automatic_reference_acceptance": False,
            "new_semantic_correctness_judgment": None,
        })

    summary = {
        "population_count": len(population),
        "core_role_record_counts": {
            role: len(index) for role, index in role_indexes.items()
        },
        "historical_audit_status_counts": dict(Counter(
            str(row.get("status")) for row in role_indexes["audit"].values()
        )),
        "historical_audit_method_counts": dict(Counter(
            str(row.get("method")) for row in role_indexes["audit"].values()
        )),
        "historical_policy_action_counts": dict(Counter(
            str(row.get("decision", {}).get("final_action"))
            for row in role_indexes["policy"].values()
        )),
        "cases_with_additional_id_linked_facts": sum(
            bool(per_case.get(cid)) for cid in population
        ),
        "cases_without_additional_id_linked_facts": sum(
            not per_case.get(cid) for cid in population
        ),
        "artifact_files_read": len(artifacts),
        "artifact_files_with_case_facts": len(files_with_case_evidence),
        "scan_issue_count": len(issues),
        "core_binding_problem_count": len(core_problems),
    }

    report = {
        "schema_version": "nb21c_test1149_offline_evidence_inventory_v1",
        "status": "OFFLINE_EVIDENCE_JOIN_COMPLETE_REQUIRES_SCOPE_REVIEW",
        "action": 26,
        "package_binding": {
            "path": package_relative,
            "sha256_file_bytes": sha(package_raw),
        },
        "core_bindings": bindings,
        "summary": summary,
        "core_binding_problems": core_problems,
        "artifact_inventory": artifacts,
        "scan_issues": issues,
        "case_records": records,
        "scope": {
            "artifact_scan_roots": output_roots,
            "maximum_file_bytes": max_bytes,
            "excluded_raw_payload_names": sorted(excluded_names),
            "discovery_is_project_wide": False,
            "historical_population_is_live_qwen_evaluation": False,
            "all_review_files_guaranteed_discovered": False,
        },
        "interpretation": [
            "Historical audit/support exists for every anchored case.",
            "Fact occurrences and artifact counts are not independent review counts.",
            "Repeated copies and reports may describe the same original review.",
            "ID association alone does not verify question/evidence/response version.",
            "Missing linked facts do not prove a case was never reviewed.",
            "File-level acceptance statements are not automatically case acceptance.",
            "Independent qualification is recorded as a claim, not inferred.",
            "Human versus API provenance requires inspection of stored method fields.",
            "No new semantic judgments or official performance metrics are produced.",
            "Need for new adjudication remains undecided until evidence scope review.",
        ],
        "new_api_calls": 0,
        "historical_code_executed": False,
        "new_semantic_judgments": 0,
        "official_no_answer_metrics_computed": False,
        "action_26_completed": False,
        "checklist_modified": False,
    }

    raw = encode(report)
    digest = sha(raw)
    directory = safe_path(
        output_roots[1] + "/NB21C.C006.test1149_offline_evidence_inventory"
    )
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f"C006.report.{digest}.json"

    fd, temporary = tempfile.mkstemp(dir=directory, prefix=".C006-")
    try:
        with os.fdopen(fd, "wb") as stream:
            stream.write(raw)
            stream.flush()
            os.fsync(stream.fileno())
        try:
            os.link(temporary, target)
        except FileExistsError:
            need(target.read_bytes() == raw, "IMMUTABLE_REPORT_CONFLICT")
    finally:
        Path(temporary).unlink(missing_ok=True)

    need(sha(target.read_bytes()) == digest, "REPORT_RELOAD_FAILED")

    result = {
        "status": report["status"],
        **summary,
        "report_path": target.relative_to(root).as_posix(),
        "report_sha256_file_bytes": digest,
        "reload_validation": "PASS",
        "new_api_calls": 0,
        "new_adjudication_required": "NOT_YET_DETERMINED",
        "action_26_completed": False,
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result


NB21C_C006_RESULT = nb21c_c006_run()

C006: core population bound: 1149; stored artifacts queued: 82250
C006: scanned 500/82250
C006: scanned 1000/82250
C006: scanned 1500/82250
C006: scanned 2000/82250
C006: scanned 2500/82250
C006: scanned 3000/82250
C006: scanned 3500/82250
C006: scanned 4000/82250
C006: scanned 4500/82250
C006: scanned 5000/82250
C006: scanned 5500/82250
C006: scanned 6000/82250
C006: scanned 6500/82250
C006: scanned 7000/82250
C006: scanned 7500/82250
C006: scanned 8000/82250
C006: scanned 8500/82250
C006: scanned 9000/82250
C006: scanned 9500/82250
C006: scanned 10000/82250
C006: scanned 10500/82250
C006: scanned 11000/82250
C006: scanned 11500/82250
C006: scanned 12000/82250
C006: scanned 12500/82250
C006: scanned 13000/82250
C006: scanned 13500/82250
C006: scanned 14000/82250
C006: scanned 14500/82250
C006: scanned 15000/82250
C006: scanned 15500/82250
C006: scanned 16000/82250
C006: scanned 16500/82250
C006: scanned 17000/82250
C006: scanned 17500/82250
C006: scanned 18000/82250
C006: scanned 1850

<div dir="rtl" align="right">

## NB21C.C007 — استخراج بسته‌های کوچک از گزارش C006

- کنترل SHA256 گزارش اصلی.
- استخراج خلاصه، مشکلات اسکن و مشخصات منابع اصلی.
- استخراج اطلاعات داوری‌های تاریخی و ارجاعات داوری هر پرونده.
- نگهداری اطلاعات سطح فایل برای بررسی روش و دامنهٔ پذیرش.
- تقسیم خروجی به ZIPهای مستقل با حجم هدف حداکثر ۱۵ مگابایت.

این استخراج داوری تازه، API، اسکن مجدد یا تغییر چک‌لیست ندارد.
گزارش اصلی تغییر نمی‌کند. فایل‌های استخراج‌شده برای بررسی هستند،
نه تأیید خودکار صحت پاسخ‌ها یا پذیرش اقدام ۲۶.

</div>

In [9]:
# NB21C.C007 — Extract compact, independently readable ZIP packages from C006.
# Reads the saved report only. No API or artifact rescan.

def nb21c_c007_run():
    import os
    import json
    import hashlib
    import tempfile
    import zipfile
    import subprocess
    from pathlib import Path
    from collections import Counter

    EXPECTED_SHA256 = (
        "8d6059d0ec83417790b31453964c9dcc39aac40122a3b2d1f0d35ab92aa4aa91"
    )
    REPORT_RELATIVE = (
        "artifacts/phase2/evaluation_and_ablation/work/outputs/"
        "NB21C.C006.test1149_offline_evidence_inventory/"
        f"C006.report.{EXPECTED_SHA256}.json"
    )
    TARGET_ZIP_BYTES = 15 * 1024 * 1024

    def need(ok, message):
        if not ok:
            raise RuntimeError(message)

    def sha(raw):
        return hashlib.sha256(raw).hexdigest()

    def encode(value):
        return json.dumps(
            value, ensure_ascii=False, sort_keys=True,
            separators=(",", ":"), allow_nan=False,
        ).encode("utf-8")

    root = Path(subprocess.run(
        ["git", "--no-optional-locks", "rev-parse", "--show-toplevel"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()).resolve()

    source = root / REPORT_RELATIVE
    need(source.is_file(), "C006_REPORT_NOT_FOUND")
    need(not source.is_symlink(), "SOURCE_SYMLINK_NOT_ALLOWED")

    print("C007: checking C006 file hash...", flush=True)
    digest = hashlib.sha256()
    with source.open("rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(block)
    need(digest.hexdigest() == EXPECTED_SHA256, "C006_HASH_MISMATCH")

    print("C007: loading saved report; this may take a moment...", flush=True)
    with source.open("r", encoding="utf-8") as stream:
        report = json.load(stream)

    cases = report.get("case_records")
    artifacts = report.get("artifact_inventory")
    need(isinstance(cases, list), "CASE_RECORDS_MISSING")
    need(isinstance(artifacts, list), "ARTIFACT_INVENTORY_MISSING")
    need(len(cases) == 1149, "CASE_COUNT_NOT_1149")
    ids = [row["candidate_id"] for row in cases]
    need(len(set(ids)) == 1149, "DUPLICATE_CASE_ID")

    destination = source.parent / ("C007.extract." + EXPECTED_SHA256[:16])
    destination.mkdir(parents=True, exist_ok=True)

    def save_exact(path, raw):
        if path.exists():
            need(path.read_bytes() == raw, "EXISTING_OUTPUT_DIFFERS:" + path.name)
            return
        fd, temporary = tempfile.mkstemp(dir=destination, prefix=".C007-")
        try:
            with os.fdopen(fd, "wb") as stream:
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())
            try:
                os.link(temporary, path)
            except FileExistsError:
                need(path.read_bytes() == raw, "OUTPUT_CONFLICT:" + path.name)
        finally:
            Path(temporary).unlink(missing_ok=True)
        need(path.read_bytes() == raw, "OUTPUT_RELOAD_FAILED:" + path.name)

    packages = []
    inventories = []

    def make_zip(kind, rows):
        """One JSONL member per ZIP; oversized batches split recursively."""
        part_counter = [0]

        def write_batch(batch):
            if not batch:
                return
            fd, temporary = tempfile.mkstemp(
                dir=destination, prefix=".C007-", suffix=".zip",
            )
            os.close(fd)
            temporary = Path(temporary)
            try:
                member_name = kind + ".jsonl"
                info = zipfile.ZipInfo(member_name, date_time=(1980, 1, 1, 0, 0, 0))
                info.compress_type = zipfile.ZIP_DEFLATED

                with zipfile.ZipFile(
                    temporary, "w", compression=zipfile.ZIP_DEFLATED,
                    compresslevel=6, allowZip64=True,
                ) as archive:
                    with archive.open(info, "w", force_zip64=True) as member:
                        for row in batch:
                            member.write(encode(row) + b"\n")

                if temporary.stat().st_size > TARGET_ZIP_BYTES and len(batch) > 1:
                    middle = len(batch) // 2
                    write_batch(batch[:middle])
                    write_batch(batch[middle:])
                    return

                with zipfile.ZipFile(temporary, "r") as archive:
                    need(archive.testzip() is None, "ZIP_CRC_FAILED")
                    with archive.open(member_name) as member:
                        count = sum(1 for line in member if line.strip())
                    need(count == len(batch), "ZIP_RECORD_COUNT_MISMATCH")

                raw = temporary.read_bytes()
                file_hash = sha(raw)
                part_counter[0] += 1
                filename = (
                    f"C007.{kind}.part{part_counter[0]:03d}."
                    f"{file_hash[:12]}.zip"
                )
                target = destination / filename
                save_exact(target, raw)
                packages.append({
                    "kind": kind,
                    "filename": filename,
                    "records": len(batch),
                    "bytes": len(raw),
                    "sha256_file_bytes": file_hash,
                    "exceeds_target_size": len(raw) > TARGET_ZIP_BYTES,
                })
            finally:
                temporary.unlink(missing_ok=True)

        # Modest starting batches avoid building one huge compressed archive.
        for start in range(0, len(rows), 100):
            write_batch(rows[start:start + 100])

    # Preserve full historical audit/support in separate packages.
    # Keep the index small and preserve additional facts without duplication.
    history_rows = []
    linked_rows = []

    for row in cases:
        cid = row["candidate_id"]
        facts = row.get("additional_artifact_facts", [])
        inventories.append({
            key: value for key, value in row.items()
            if key not in {
                "historical_audit", "historical_support",
                "additional_artifact_facts",
            }
        })
        history_rows.append({
            "candidate_id": cid,
            "historical_audit": row.get("historical_audit"),
            "historical_support": row.get("historical_support"),
        })
        linked_rows.append({
            "candidate_id": cid,
            "additional_artifact_facts": facts,
        })

    # Only omit inventory rows containing no extracted facts.
    # The counts of those omitted rows are recorded explicitly.
    artifact_rows = [
        row for row in artifacts
        if row.get("file_level_recorded_facts")
        or row.get("case_fact_occurrences_not_unique_reviews", 0)
        or row.get("parse_failures", 0)
    ]

    overview = {
        "schema_version": "nb21c_c007_extraction_overview_v1",
        "source_report": {
            "path": REPORT_RELATIVE,
            "sha256_file_bytes": EXPECTED_SHA256,
        },
        "source_status": report.get("status"),
        "summary": report.get("summary"),
        "core_bindings": report.get("core_bindings"),
        "package_binding": report.get("package_binding"),
        "core_binding_problems": report.get("core_binding_problems", []),
        "scan_issues": report.get("scan_issues", []),
        "scope": report.get("scope"),
        "interpretation": report.get("interpretation"),
        "extraction_counts": {
            "case_count": len(cases),
            "historical_review_records": len(history_rows),
            "cases_with_additional_facts": sum(
                bool(row["additional_artifact_facts"]) for row in linked_rows
            ),
            "additional_fact_occurrences": sum(
                len(row["additional_artifact_facts"]) for row in linked_rows
            ),
            "original_artifact_inventory_count": len(artifacts),
            "extracted_artifact_inventory_count": len(artifact_rows),
            "omitted_inventory_rows_without_extracted_facts":
                len(artifacts) - len(artifact_rows),
        },
        "new_api_calls": 0,
        "new_semantic_judgments": 0,
        "action_26_completed": False,
    }

    save_exact(destination / "C007.overview.json", encode(overview))
    save_exact(
        destination / "C007.case_index.jsonl",
        b"".join(encode(row) + b"\n" for row in inventories),
    )

    print("C007: packaging historical reviews...", flush=True)
    make_zip("historical_reviews", history_rows)

    print("C007: packaging linked review facts...", flush=True)
    make_zip("linked_facts", linked_rows)

    print("C007: packaging artifact scope and acceptance facts...", flush=True)
    make_zip("artifact_inventory", artifact_rows)

    manifest = {
        "schema_version": "nb21c_c007_extraction_manifest_v1",
        "source_report_sha256_file_bytes": EXPECTED_SHA256,
        "package_count": len(packages),
        "packages": packages,
        "record_totals_by_kind": dict(Counter({
            kind: sum(item["records"] for item in packages if item["kind"] == kind)
            for kind in {item["kind"] for item in packages}
        })),
        "source_report_unchanged": True,
        "new_api_calls": 0,
    }
    need(
        manifest["record_totals_by_kind"].get("historical_reviews") == 1149
        and manifest["record_totals_by_kind"].get("linked_facts") == 1149
        and manifest["record_totals_by_kind"].get("artifact_inventory")
            == len(artifact_rows),
        "EXTRACTION_COVERAGE_MISMATCH",
    )
    save_exact(destination / "C007.manifest.json", encode(manifest))

    print(json.dumps({
        "status": "C007_EXTRACTION_COMPLETE",
        "output_directory": str(destination),
        "attach_first": [
            "C007.overview.json",
            "C007.manifest.json",
        ],
        "zip_package_count": len(packages),
        "total_zip_megabytes": round(
            sum(item["bytes"] for item in packages) / (1024 * 1024), 2
        ),
        "oversized_single_record_packages": [
            item["filename"] for item in packages
            if item["exceeds_target_size"]
        ],
        "coverage_validation": "PASS",
        "new_api_calls": 0,
    }, ensure_ascii=False, indent=2))

    return manifest


NB21C_C007_MANIFEST = nb21c_c007_run()

C007: checking C006 file hash...
C007: loading saved report; this may take a moment...
C007: packaging historical reviews...
C007: packaging linked review facts...
C007: packaging artifact scope and acceptance facts...
{
  "status": "C007_EXTRACTION_COMPLETE",
  "output_directory": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/evaluation_and_ablation/work/outputs/NB21C.C006.test1149_offline_evidence_inventory/C007.extract.8d6059d0ec834177",
  "attach_first": [
    "C007.overview.json",
    "C007.manifest.json"
  ],
  "zip_package_count": 717,
  "total_zip_megabytes": 47.98,
  "oversized_single_record_packages": [],
  "coverage_validation": "PASS",
  "new_api_calls": 0
}


<div dir="rtl" align="right">

## NB21C.C008 — تطبیق نهایی سوابق موجود با اقدام ۲۶

### هدف

تطبیق سوابق ۱۱۴۹ پروندهٔ Test با قرارداد پاسخ‌پذیری،
مرجع پیشنهادی و مقایسهٔ رفتار موجود؛ بدون تکرار داوری.

### بررسی‌ها

- کنترل هش گزارش C006 و منابع مشخص C190، C197 و C198.
- تطبیق جمعیت، شناسه، هش پاسخ و پوشش بررسی‌های تاریخی.
- تفکیک ۴۵ خودداری تاریخی از سایر پرونده‌ها.
- کنترل اتصال پیشنهادهای مرجع و رفتار مشاهده‌شده به همان سؤال‌ها.
- بازخوانی ادعاهای پذیرش و محدودیت‌های ثبت‌شده در گزارش تجمیعی.
- ثبت موارد نامعلوم و شواهدی که هنوز خوانده نشده‌اند.

### خروجی

گزارش نسخه‌دارِ پایان تطبیق با وضعیت واقعی اقدام ۲۶.

وجود بررسی تاریخی، صحت قطعی پاسخ را ثابت نمی‌کند.
پیشنهاد مدل بدون پذیرش روش و مرجع، معیار رسمی محسوب نمی‌شود.
فایل‌های دارای خطای اسکن نادیده گرفته نمی‌شوند.
هیچ API، داوری تازه یا تغییر خودکار چک‌لیست انجام نمی‌شود.

</div>

In [10]:
# NB21C.C008 — Reconcile existing evidence for action 26.
# No API, historical-code execution, broad rescan, or new semantic judgment.

def nb21c_c008_run():
    import os
    import json
    import hashlib
    import tempfile
    import subprocess
    from pathlib import Path
    from collections import Counter

    def need(ok, message):
        if not ok:
            raise RuntimeError(message)

    def sha(raw):
        return hashlib.sha256(raw).hexdigest()

    def encode(value):
        return json.dumps(
            value, ensure_ascii=False, sort_keys=True,
            separators=(",", ":"), allow_nan=False,
        ).encode("utf-8")

    def strict_json(raw):
        def pairs(items):
            result = {}
            for key, value in items:
                need(key not in result, "DUPLICATE_JSON_KEY")
                result[key] = value
            return result

        def reject(value):
            raise ValueError("NONSTANDARD_JSON_CONSTANT:" + value)

        return json.loads(
            raw, object_pairs_hook=pairs, parse_constant=reject,
        )

    root = Path(subprocess.run(
        ["git", "--no-optional-locks", "rev-parse", "--show-toplevel"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()).resolve()

    def safe_path(relative):
        rel = Path(relative)
        need(
            not rel.is_absolute() and ".." not in rel.parts,
            "UNSAFE_PATH",
        )
        current = root
        for part in rel.parts:
            current = current / part
            need(not current.is_symlink(), "SYMLINK_NOT_ALLOWED")
        resolved = current.resolve()
        resolved.relative_to(root)
        return resolved

    receipts = {}

    def load_bound(relative, expected):
        raw = safe_path(relative).read_bytes()
        need(sha(raw) == expected, "HASH_MISMATCH:" + relative)
        receipts[relative] = {
            "sha256_file_bytes": expected,
            "bytes": len(raw),
        }
        return strict_json(raw)

    evaluation = "artifacts/phase2/evaluation_and_ablation/work/outputs/"
    answerability = (
        "artifacts/phase2/answerability_safety_confidence/work/outputs/"
    )

    c006_hash = (
        "8d6059d0ec83417790b31453964c9dcc39aac40122a3b2d1f0d35ab92aa4aa91"
    )
    c006 = load_bound(
        evaluation
        + "NB21C.C006.test1149_offline_evidence_inventory/"
        + f"C006.report.{c006_hash}.json",
        c006_hash,
    )

    def index_rows(rows, field, label):
        need(isinstance(rows, list), label + ":ROWS_NOT_LIST")
        result = {}
        for row in rows:
            need(isinstance(row, dict), label + ":ROW_NOT_OBJECT")
            cid = row.get(field)
            need(isinstance(cid, str) and cid, label + ":ID_INVALID")
            need(cid not in result, label + ":DUPLICATE_ID")
            result[cid] = row
        return result

    cases = index_rows(c006["case_records"], "candidate_id", "C006")
    need(len(cases) == 1149, "C006_POPULATION_NOT_1149")

    artifact_inventory = index_rows(
        c006["artifact_inventory"], "path", "ARTIFACT_INVENTORY",
    )

    # Load one exact C190 artifact already registered by C006.
    # No directory discovery or selection by modification time.
    c190_paths = [
        path for path in artifact_inventory
        if path.startswith(
            answerability + "nb20c_behavior_reference_C190_v1/"
        )
        and Path(path).name.startswith("C190.reference_plan.")
        and path.endswith(".json")
    ]
    need(len(c190_paths) == 1, "C190_NOT_UNIQUE_IN_SAVED_INVENTORY")
    c190_path = c190_paths[0]
    c190 = load_bound(
        c190_path,
        artifact_inventory[c190_path]["sha256_file_bytes"],
    )

    c197_hash = (
        "1a33dbdd0cb352794f7f3fa6228f0279f4a1d832cb573bea9e802211d2dcfc12"
    )
    c198_hash = (
        "c0fbe625c12abfda2f57d7441fe2d575c2fd0457a1dd2061c909297bc9562b6a"
    )
    c181_hash = (
        "9e5988d4cc7e50e8862dd81929ee09f84693ead0a6c97232581943de78c100aa"
    )
    c189_hash = (
        "a31fd52a5583fabe13fd4ab756adb0119280b37b71bce3379ce6ce1d1f8639b3"
    )

    c197 = load_bound(
        answerability + "nb20c_model_reference_registry_C197_v1/"
        + f"C197.references.{c197_hash}.json",
        c197_hash,
    )
    c198 = load_bound(
        answerability + "nb20c_abstention_comparison_C198_v1/"
        + f"C198.comparison.{c198_hash}.json",
        c198_hash,
    )
    c181 = load_bound(
        answerability + "nb20c_historical_abstention_C181_v1/"
        + f"C181.inventory.{c181_hash}.json",
        c181_hash,
    )
    c189 = load_bound(
        answerability + "nb20c_behavior_consolidation_C189_v1/"
        + f"C189.report.{c189_hash}.json",
        c189_hash,
    )

    findings = []

    def check(ok, name, detail=None):
        item = {"check": name, "passed": bool(ok)}
        if detail is not None:
            item["detail"] = detail
        findings.append(item)
        return bool(ok)

    dimensions = (
        "abstention_justified",
        "ambiguity_conflict",
        "evidence_sufficiency",
        "factual_claim_support",
        "outside_details",
        "question_answer_fit",
        "scope_modality_negation_quantity",
    )

    historical_counts = Counter()
    method_counts = Counter()
    dimension_coverage = Counter()
    historical_abstentions = set()
    case_binding_issues = []
    historical_qualification_profiles = Counter()

    for cid, row in cases.items():
        audit = row.get("historical_audit", {})
        support = row.get("historical_support", {})
        historical_counts[str(audit.get("status"))] += 1
        method_counts[str(audit.get("method"))] += 1

        if audit.get("status") == "VALID_ABSTENTION":
            historical_abstentions.add(cid)

        issues = []
        if audit.get("candidate_id") != cid:
            issues.append("AUDIT_ID_MISMATCH")
        if support.get("candidate_id") != cid:
            issues.append("SUPPORT_ID_MISMATCH")

        ah = audit.get("response_sha256")
        sh = support.get("response_sha256")
        active_hash = row.get("active_response_sha256_recorded")
        if not (
            isinstance(ah, str) and len(ah) == 64
            and ah == sh == active_hash
        ):
            issues.append("AUDIT_SUPPORT_ACTIVE_RESPONSE_HASH_MISMATCH")

        for field in dimensions:
            value = audit.get("dimensions", {}).get(field)
            if isinstance(value, str) and value.strip():
                dimension_coverage[field] += 1
            else:
                issues.append("MISSING_REVIEW_DIMENSION:" + field)

        profile = (
            str(audit.get("context_independence")),
            str(support.get("independent")),
            str(support.get("independent_semantic_entailment")),
        )
        historical_qualification_profiles[profile] += 1

        if issues:
            case_binding_issues.append({
                "candidate_id": cid, "issues": issues,
            })

    check(
        not c006.get("core_binding_problems"),
        "C006_CORE_BINDINGS",
        c006.get("core_binding_problems"),
    )
    check(not case_binding_issues, "HISTORICAL_CASE_BINDINGS_AND_DIMENSIONS")
    check(
        dict(historical_counts) == {
            "KEEP": 836, "REPAIRED": 224,
            "REVIEW_REQUIRED": 44, "VALID_ABSTENTION": 45,
        },
        "HISTORICAL_STATUS_COUNTS",
        dict(historical_counts),
    )

    originals = index_rows(c181["cases"], "candidate_id", "C181")
    reference_inputs = index_rows(
        c190["reference_inputs"], "candidate_id", "C190",
    )
    proposals = index_rows(
        c197["reference_proposals"], "candidate_id", "C197",
    )
    observed = index_rows(
        c189["case_observations"], "candidate_id", "C189_OBSERVED",
    )
    unknown = index_rows(
        c189["unknown_outcome_cases"], "candidate_id", "C189_UNKNOWN",
    )
    comparisons = index_rows(
        c198["case_comparisons"], "candidate_id", "C198",
    )

    target_ids = set(originals)
    check(
        len(target_ids) == 45
        and target_ids == historical_abstentions,
        "ALL_45_TARGETS_ARE_EXACT_HISTORICAL_ABSTENTION_GROUP",
    )
    check(
        target_ids == set(reference_inputs) == set(proposals)
        and target_ids <= set(cases),
        "C181_C190_C197_POPULATION_BINDING",
    )
    check(
        len(observed) == 44 and len(unknown) == 1
        and not (set(observed) & set(unknown))
        and set(observed) | set(unknown) == target_ids
        and set(comparisons) == set(observed),
        "OBSERVED_UNKNOWN_COMPARISON_POPULATION_BINDING",
    )

    contract = c190["reference_contract"]
    contract_hash = sha(encode(contract))
    check(
        contract_hash == c190.get("reference_contract_sha256")
        == c197.get("source_reference_contract_sha256"),
        "C190_C197_CONTRACT_HASH_BINDING",
    )
    check(
        contract.get("evaluation_view")
        == "QUESTION_AND_EXACT_ORIGINAL_EVIDENCE",
        "CONTRACT_USES_ORIGINAL_EXECUTION_EVIDENCE",
    )
    check(
        c198.get("source_file_sha256", {}).get("C181") == c181_hash
        and c198.get("source_file_sha256", {}).get("C189") == c189_hash
        and c198.get("source_file_sha256", {}).get("C197") == c197_hash,
        "C198_INPUT_HASH_BINDINGS",
    )

    question_issues = []
    matrix = {}
    case_results = []

    for cid in sorted(target_ids):
        original = originals[cid]
        ref_input = reference_inputs.get(cid, {})
        proposal = proposals.get(cid, {})
        question = original.get("question")
        qhash = (
            sha(question.encode("utf-8"))
            if isinstance(question, str) else None
        )
        issues = []

        if not (
            qhash is not None
            and qhash == original.get("query_sha256")
            == ref_input.get("query_sha256")
            == proposal.get("query_sha256")
            and question == ref_input.get("question")
        ):
            issues.append("QUESTION_BINDING_MISMATCH")

        if encode(original.get("original_evidence")) != encode(
            ref_input.get("original_evidence")
        ):
            issues.append("ORIGINAL_EVIDENCE_BINDING_MISMATCH")

        historical_hash = cases.get(cid, {}).get(
            "active_response_sha256_recorded"
        )
        if historical_hash != original.get("active_response_sha256"):
            issues.append("C181_HISTORICAL_ACTIVE_RESPONSE_MISMATCH")

        behavior = observed.get(cid, {}).get("response_behavior")
        reference = proposal.get("reference_behavior")

        if cid in observed:
            saved = comparisons.get(cid, {})
            if not (
                saved.get("observed_behavior") == behavior
                and saved.get("proposed_reference_behavior") == reference
                and saved.get("query_sha256") == qhash
            ):
                issues.append("C198_CASE_COMPARISON_MISMATCH")
            matrix.setdefault(behavior, Counter())[reference] += 1

        if issues:
            question_issues.append({
                "candidate_id": cid, "issues": issues,
            })

        case_results.append({
            "candidate_id": cid,
            "query_sha256": qhash,
            "historical_audit_status":
                cases.get(cid, {}).get("historical_audit", {}).get("status"),
            "observed_behavior": behavior,
            "outcome_unknown": cid in unknown,
            "model_proposed_reference_behavior": reference,
            "binding_issues": issues,
            "verified_semantic_error": None,
        })

    matrix = {key: dict(value) for key, value in matrix.items()}
    check(not question_issues, "TARGET_QUESTION_EVIDENCE_RESPONSE_BINDINGS")
    check(
        matrix == c198.get("observed_by_proposed_reference"),
        "RECOMPUTED_MATRIX_MATCHES_C198",
        matrix,
    )

    # Preserve all extracted qualification/acceptance claims with their scope.
    # No chronological order or supersession is inferred from cell numbers.
    claim_keys = {
        "formal_acceptance", "acceptance_result",
        "operational_acceptance_established",
        "contract_acceptance_established",
        "reference_qualification_established",
        "qualified_independent_reference_established",
        "reference_semantic_correctness_independently_verified",
        "independent_reference_established",
        "reference_established_in_model_proposal_scope",
        "action_26_completed", "action26_completed",
    }

    scoped_claims = {}
    for path, artifact in artifact_inventory.items():
        for fact in artifact.get("file_level_recorded_facts", []):
            fields = fact.get("recorded_fields", {})
            selected = {
                key: value for key, value in fields.items()
                if key in claim_keys
            }
            if selected:
                item = {
                    "path": path,
                    "location": fact.get("location"),
                    "recorded_claims": selected,
                    "context_fields": fields,
                    "case_id": None,
                }
                scoped_claims[sha(encode(item))] = item

    for cid, row in cases.items():
        for fact in row.get("additional_artifact_facts", []):
            fields = fact.get("recorded_fields", {})
            selected = {
                key: value for key, value in fields.items()
                if key in claim_keys
            }
            if selected:
                item = {
                    "path": fact.get("path"),
                    "location": fact.get("location"),
                    "recorded_claims": selected,
                    "context_fields": fields,
                    "case_id": cid,
                }
                scoped_claims[sha(encode(item))] = item

    # Candidate claims are a review queue, never automatic acceptance.
    positive_candidates = [
        item for item in scoped_claims.values()
        if any(
            value is True or (
                isinstance(value, str)
                and value.upper() in {"PASS", "ACCEPTED", "ESTABLISHED"}
            )
            for value in item["recorded_claims"].values()
        )
    ]

    scan_issues = c006.get("scan_issues", [])
    failed_checks = [item for item in findings if not item["passed"]]
    technical_pass = not failed_checks

    qualification = {
        "contract_status": contract.get("status"),
        "contract_acceptance_established":
            c190.get("contract_acceptance_established"),
        "C197_qualified_independent_reference_established":
            c197.get("qualified_independent_reference_established"),
        "C197_reference_semantic_correctness_independently_verified":
            c197.get("reference_semantic_correctness_independently_verified"),
        "C198_operational_acceptance_established":
            c198.get("operational_acceptance_established"),
    }

    remaining = []
    if failed_checks:
        remaining.append("RESOLVE_SPECIFIC_FAILED_BINDINGS")
    if scan_issues:
        remaining.append("RESOLVE_OR_JUSTIFY_THE_12_SAVED_SCAN_EXCEPTIONS")
    remaining.extend([
        "REVIEW_SCOPED_ACCEPTANCE_CLAIMS_FOR_APPLICABILITY_AND_SUPERSESSION",
        "ESTABLISH_ACCEPTED_REFERENCE_AND_METRIC_SCOPE_BEFORE_OFFICIAL_SCORING",
    ])

    result = {
        "schema_version": "nb21c_action26_reconciliation_C008_v1",
        "status": (
            "EXISTING_EVIDENCE_RECONCILED_ACCEPTANCE_NOT_ESTABLISHED"
            if technical_pass else
            "RECONCILIATION_RECORDED_WITH_BINDING_FAILURES"
        ),
        "action": 26,
        "input_receipts": receipts,
        "validation_checks": findings,
        "failed_checks": failed_checks,
        "historical_population": {
            "count": len(cases),
            "status_counts": dict(historical_counts),
            "method_counts": dict(method_counts),
            "dimension_coverage": dict(dimension_coverage),
            "case_binding_issues": case_binding_issues,
            "qualification_profiles": [
                {
                    "context_independence": key[0],
                    "support_independent": key[1],
                    "independent_semantic_entailment": key[2],
                    "count": count,
                }
                for key, count in historical_qualification_profiles.items()
            ],
            "historical_reviews_exist": True,
            "blanket_readjudication_authorized": False,
        },
        "targeted_comparison": {
            "target_count": len(originals),
            "observed_count": len(observed),
            "unknown_count": len(unknown),
            "model_reference_counts": dict(Counter(
                row.get("reference_behavior") for row in proposals.values()
            )),
            "observed_by_model_reference": matrix,
            "question_evidence_binding_issues": question_issues,
            "cases": case_results,
            "comparison_is_official_accuracy": False,
        },
        "reference_contract": contract,
        "recorded_qualification": qualification,
        "scoped_acceptance_claims": list(scoped_claims.values()),
        "positive_claim_candidates_requiring_scope_review":
            positive_candidates,
        "saved_scan_exceptions": scan_issues,
        "scan_exception_type_counts": dict(Counter(
            item.get("reason") for item in scan_issues
        )),
        "remaining_work": remaining,
        "official_no_answer_metrics": {
            "precision": None, "recall": None, "f1": None,
            "status": "NOT_ESTABLISHED_BY_THIS_RECONCILIATION",
        },
        "reconciliation_round_finished": True,
        "technical_reconciliation_pass": technical_pass,
        "action_26_completed": False,
        "checklist_modified": False,
        "new_api_calls": 0,
        "new_semantic_judgments": 0,
        "need_for_new_api": "NOT_ESTABLISHED",
        "limitations": [
            "Historical judgments are preserved, not independently revalidated.",
            "Stored acceptance claims do not establish applicability by themselves.",
            "Unknown outcome remains unknown.",
            "Historical targeted results do not establish population accuracy.",
            "Saved scan exceptions remain visible; completeness is not asserted.",
            "This cell does not infer that no usable reference exists project-wide.",
        ],
    }

    directory = safe_path(
        evaluation + "NB21C.C008.action26_reconciliation"
    )
    directory.mkdir(parents=True, exist_ok=True)
    raw = encode(result)
    digest = sha(raw)
    target = directory / f"C008.report.{digest}.json"

    fd, temporary = tempfile.mkstemp(dir=directory, prefix=".C008-")
    try:
        with os.fdopen(fd, "wb") as stream:
            stream.write(raw)
            stream.flush()
            os.fsync(stream.fileno())
        try:
            os.link(temporary, target)
        except FileExistsError:
            need(target.read_bytes() == raw, "IMMUTABLE_REPORT_CONFLICT")
    finally:
        Path(temporary).unlink(missing_ok=True)

    need(sha(target.read_bytes()) == digest, "REPORT_RELOAD_FAILED")

    summary = {
        "status": result["status"],
        "technical_reconciliation_pass": technical_pass,
        "population_count": len(cases),
        "historical_review_counts": dict(historical_counts),
        "target_count": len(originals),
        "observed_count": len(observed),
        "unknown_count": len(unknown),
        "observed_by_model_reference": matrix,
        "recorded_qualification": qualification,
        "failed_checks": failed_checks,
        "saved_scan_exception_count": len(scan_issues),
        "positive_claim_candidate_count": len(positive_candidates),
        "remaining_work": remaining,
        "new_api_calls": 0,
        "need_for_new_api": "NOT_ESTABLISHED",
        "action_26_completed": False,
        "report_path": target.relative_to(root).as_posix(),
        "report_sha256_file_bytes": digest,
        "reload_validation": "PASS",
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary


NB21C_C008_RESULT = nb21c_c008_run()

{
  "status": "EXISTING_EVIDENCE_RECONCILED_ACCEPTANCE_NOT_ESTABLISHED",
  "technical_reconciliation_pass": true,
  "population_count": 1149,
  "historical_review_counts": {
    "KEEP": 836,
    "REPAIRED": 224,
    "REVIEW_REQUIRED": 44,
    "VALID_ABSTENTION": 45
  },
  "target_count": 45,
  "observed_count": 44,
  "unknown_count": 1,
  "observed_by_model_reference": {
    "ABSTENTION_ONLY": {
      "CLARIFICATION_REQUIRED": 6,
      "ANSWER_SUPPORTED": 28,
      "BOUNDED_ANSWER_SUPPORTED": 3
    },
    "CLARIFICATION_ONLY": {
      "CLARIFICATION_REQUIRED": 1,
      "ANSWER_SUPPORTED": 1
    },
    "SUBSTANTIVE_RESPONSE": {
      "ANSWER_SUPPORTED": 3,
      "BOUNDED_ANSWER_SUPPORTED": 2
    }
  },
  "recorded_qualification": {
    "contract_status": "ENGINEERING_PROPOSAL",
    "contract_acceptance_established": false,
    "C197_qualified_independent_reference_established": false,
    "C197_reference_semantic_correctness_independently_verified": false,
    "C198_operational_acceptan

<div dir="rtl" align="right">

## NB21C.C009 — رسیدگی به ۱۲ استثنای باقی‌ماندهٔ اقدام ۲۶

### هدف

بررسی همان ۱۲ فایلی که در C006 کامل پردازش نشدند،
پیش از عبور از اقدام ۲۶.

### اجرا

- دریافت فهرست دقیق استثناها از گزارش معتبر C008.
- خواندن ۱۰ فایل بزرگ، بدون سقف حجم قبلی.
- اصلاح پردازش case_id دیکشنری و خواندن C192 و C195.
- استخراج وضعیت‌ها، روش داوری، اطلاعات مرجع و ادعاهای پذیرش.
- تفکیک خطای ابزار بررسی از خطای احتمالی داده.
- ثبت موفقیت یا شکست هر فایل، همراه هش و علت.

### تصمیم ادامه

پس از یک دور رسیدگی به هر ۱۲ مورد، نتیجه ثبت و ادامهٔ مستقل
به اقدام ۲۷ منتقل می‌شود؛ حتی اگر محدودیتی باقی مانده باشد.

عبور به اقدام بعد به معنی پذیرش معیار رسمی، رفع تمام موانع،
یا تیک خودکار اقدام ۲۶ نیست. هیچ API، داوری تازه،
اجرای کد تاریخی یا تغییر چک‌لیست انجام نمی‌شود.

</div>

In [11]:
# NB21C.C009 — Address all 12 saved scan exceptions, then advance.
# Replaces the previously proposed, unexecuted C009 closure cell.
# No API, historical execution, broad discovery, or new semantic judgment.

def nb21c_c009_run():
    import os
    import gc
    import json
    import hashlib
    import tempfile
    import subprocess
    from pathlib import Path
    from collections import Counter, defaultdict

    def need(ok, message):
        if not ok:
            raise RuntimeError(message)

    def sha(raw):
        return hashlib.sha256(raw).hexdigest()

    def encode(value):
        return json.dumps(
            value, ensure_ascii=False, sort_keys=True,
            separators=(",", ":"), allow_nan=False,
        ).encode("utf-8")

    def strict_json(raw):
        def pairs(items):
            result = {}
            for key, value in items:
                need(key not in result, "DUPLICATE_JSON_KEY")
                result[key] = value
            return result

        def reject(value):
            raise ValueError("NONSTANDARD_JSON_CONSTANT:" + value)

        return json.loads(
            raw, object_pairs_hook=pairs, parse_constant=reject,
        )

    root = Path(subprocess.run(
        ["git", "--no-optional-locks", "rev-parse", "--show-toplevel"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()).resolve()

    def safe_path(relative):
        rel = Path(relative)
        need(
            not rel.is_absolute() and ".." not in rel.parts,
            "UNSAFE_PATH",
        )
        current = root
        for part in rel.parts:
            current = current / part
            need(not current.is_symlink(), "SYMLINK_NOT_ALLOWED")
        resolved = current.resolve()
        resolved.relative_to(root)
        return resolved

    prefix = "artifacts/phase2/evaluation_and_ablation/work/outputs/"
    c008_hash = (
        "5aff66235bcdb346f09d51532836412a5c6f4a6ab5b8ef7cccb01719dd5c645b"
    )
    c008_relative = (
        prefix + "NB21C.C008.action26_reconciliation/"
        + f"C008.report.{c008_hash}.json"
    )
    source_raw = safe_path(c008_relative).read_bytes()
    need(sha(source_raw) == c008_hash, "C008_HASH_MISMATCH")
    source = strict_json(source_raw)
    del source_raw

    need(source.get("action") == 26, "SOURCE_ACTION_MISMATCH")
    need(
        source.get("technical_reconciliation_pass") is True,
        "C008_RECONCILIATION_NOT_PASS",
    )

    exceptions = source["saved_scan_exceptions"]
    need(len(exceptions) == 12, "EXPECTED_EXACTLY_12_EXCEPTIONS")
    paths = [item["path"] for item in exceptions]
    need(len(set(paths)) == 12, "DUPLICATE_EXCEPTION_PATH")

    # Retain the established population, without loading C006 again.
    population = None
    population_binding = source["historical_population"]["count"]
    need(population_binding == 1149, "POPULATION_COUNT_CHANGED")

    # Read the already-bound historical policy to obtain the exact ID set.
    # Its path/hash are established in this conversation and C006.
    policy_relative = (
        "artifacts/phase2/answerability_safety_confidence/work/outputs/"
        "nb20_test_policy_records_chatgpt_recovery_v1.jsonl"
    )
    policy_expected = (
        "7363fca9ce6aa30e1020fb49fab8782fb1c87c4d3a3691dd3684bcf419dbf1ac"
    )
    policy_raw = safe_path(policy_relative).read_bytes()
    need(sha(policy_raw) == policy_expected, "POLICY_HASH_MISMATCH")
    population = set()
    for line in policy_raw.splitlines():
        if not line.strip():
            continue
        row = strict_json(line)
        cid = row.get("candidate_id")
        if not isinstance(cid, str):
            decision = row.get("decision")
            cid = (
                decision.get("candidate_id")
                if isinstance(decision, dict) else None
            )
        need(isinstance(cid, str), "POLICY_ID_INVALID")
        need(cid not in population, "POLICY_DUPLICATE_ID")
        population.add(cid)
    need(len(population) == 1149, "POLICY_ID_COUNT_CHANGED")
    del policy_raw

    # These fields identify claims worth reviewing.
    # Their values are never treated as automatic acceptance.
    acceptance_keys = {
        "formal_acceptance", "acceptance_result",
        "operational_acceptance_established",
        "contract_acceptance_established",
        "reference_qualification_established",
        "qualified_independent_reference_established",
        "reference_semantic_correctness_independently_verified",
        "independent_reference_established",
        "reference_established_in_model_proposal_scope",
        "action_26_completed", "action26_completed",
    }
    profile_keys = acceptance_keys | {
        "status", "method", "reference_method", "reviewer_identity",
        "model", "reference_behavior", "response_behavior",
        "answerability", "answerability_label",
        "independent", "context_independence",
        "independent_semantic_entailment", "verified_ground_truth",
        "reference_status", "reference_qualification_status",
        "final_action", "sufficiency", "evidence_sufficiency",
        "full_policy_evaluation_complete",
    }
    context_keys = {
        "status", "schema_version", "requirement_id",
        "checklist_action_number", "action",
        "completion_scope", "metric_scope", "evaluation_view",
        "input_condition", "reference_method", "reviewer_identity",
        "model", "query_sha256", "response_sha256",
        "active_response_sha256", "source_reference_contract_sha256",
    }

    def scalar(value):
        return value is None or isinstance(value, (str, bool, int, float))

    def inspect_document(doc, state, start_pointer):
        # Iterative traversal avoids recursion-depth failures.
        stack = [(doc, start_pointer, None)]

        while stack:
            value, pointer, owner = stack.pop()
            state["visited_nodes"] += 1

            if isinstance(value, dict):
                cid = value.get("candidate_id")
                case_id = value.get("case_id")

                # FIX: never test dict/list membership in a set.
                if isinstance(case_id, dict):
                    state["dict_case_id_count"] += 1

                explicit = (
                    cid if isinstance(cid, str)
                    else case_id if isinstance(case_id, str)
                    else None
                )
                if explicit is not None:
                    owner = explicit if explicit in population else None
                    if owner is not None:
                        state["linked_ids"].add(owner)

                claims = {
                    key: value[key] for key in acceptance_keys
                    if key in value and scalar(value[key])
                }
                if claims:
                    state["acceptance_claims"].append({
                        "pointer": pointer,
                        "candidate_id": owner,
                        "claims": claims,
                        "context": {
                            key: value[key] for key in context_keys
                            if key in value and scalar(value[key])
                        },
                        "scope_and_supersession_verified": False,
                    })

                for key, item in value.items():
                    key_text = str(key)
                    escaped = key_text.replace("~", "~0").replace("/", "~1")
                    child_pointer = pointer + "/" + escaped

                    if key_text in profile_keys and scalar(item):
                        # Count short labels exactly; retain long prose count only.
                        if isinstance(item, str) and len(item) > 240:
                            label = "<LONG_PROSE_NOT_CLASSIFIED>"
                            state["long_prose_fields"] += 1
                        else:
                            label = json.dumps(
                                item, ensure_ascii=False, sort_keys=True,
                            )
                        state["profiles"][key_text][label] += 1

                    child_owner = (
                        key_text if key_text in population else owner
                    )
                    if key_text in population:
                        state["linked_ids"].add(key_text)

                    if isinstance(item, (dict, list)):
                        stack.append((item, child_pointer, child_owner))

            elif isinstance(value, list):
                for number in range(len(value) - 1, -1, -1):
                    item = value[number]
                    if isinstance(item, (dict, list)):
                        stack.append((
                            item, pointer + "/" + str(number), owner,
                        ))

    results = []

    for number, previous in enumerate(exceptions, 1):
        relative = previous["path"]
        print(
            f"C009: {number}/12 — {Path(relative).name}",
            flush=True,
        )

        state = {
            "visited_nodes": 0,
            "dict_case_id_count": 0,
            "long_prose_fields": 0,
            "linked_ids": set(),
            "profiles": defaultdict(Counter),
            "acceptance_claims": [],
        }
        result = {
            "path": relative,
            "previous_exception": previous,
            "status": "NOT_PROCESSED",
            "parse_errors": [],
        }

        try:
            path = safe_path(relative)
            need(path.is_file(), "FILE_MISSING")
            before = path.stat()
            digest = hashlib.sha256()
            parsed_records = 0

            # No former 64 MiB limit: only these 12 explicit files are read.
            if path.suffix.lower() == ".jsonl":
                with path.open("rb") as stream:
                    for line_number, raw_line in enumerate(stream, 1):
                        digest.update(raw_line)
                        if not raw_line.strip():
                            continue
                        try:
                            doc = strict_json(raw_line)
                        except Exception as error:
                            result["parse_errors"].append({
                                "line": line_number,
                                "type": type(error).__name__,
                                "message": str(error)[:300],
                            })
                            continue
                        inspect_document(
                            doc, state, f"line:{line_number}",
                        )
                        parsed_records += 1
                        del doc
            else:
                # One large JSON file at a time; memory released between files.
                raw = path.read_bytes()
                digest.update(raw)
                doc = strict_json(raw)
                del raw
                inspect_document(doc, state, "")
                parsed_records = 1
                del doc

            after = path.stat()
            need(
                before.st_size == after.st_size
                and before.st_mtime_ns == after.st_mtime_ns,
                "FILE_CHANGED_DURING_READ",
            )
            result.update({
                "status": (
                    "READ_WITH_RECORD_ERRORS"
                    if result["parse_errors"] else "READ_AND_INSPECTED"
                ),
                "bytes": after.st_size,
                "sha256_file_bytes": digest.hexdigest(),
                "parsed_document_or_jsonl_record_count": parsed_records,
                "unchanged_during_read": True,
            })

            if "bytes" in previous:
                result["size_matches_previous_scan"] = (
                    after.st_size == previous["bytes"]
                )
                if not result["size_matches_previous_scan"]:
                    result["status"] = "READ_BUT_VERSION_SIZE_CHANGED"

        except Exception as error:
            result.update({
                "status": "READ_FAILED",
                "error_type": type(error).__name__,
                "error_message": str(error)[:500],
            })

        result.update({
            "visited_container_nodes": state["visited_nodes"],
            "dict_case_id_count_handled": state["dict_case_id_count"],
            "unique_test_ids_linked": len(state["linked_ids"]),
            "test_ids_linked": sorted(state["linked_ids"]),
            "recorded_label_profiles": {
                key: dict(counts)
                for key, counts in state["profiles"].items()
            },
            "long_prose_fields_not_classified": state["long_prose_fields"],
            "acceptance_claims_for_scope_review": state["acceptance_claims"],
            "semantic_correctness_newly_verified": False,
        })
        results.append(result)
        gc.collect()

    need(len(results) == 12, "NOT_ALL_12_ATTEMPTED")

    unresolved = [
        {
            "path": item["path"],
            "status": item["status"],
            "parse_error_count": len(item["parse_errors"]),
            "error_message": item.get("error_message"),
        }
        for item in results
        if item["status"] != "READ_AND_INSPECTED"
    ]

    # Expose every positive claim; do not infer chronological supersession.
    positive_claims = []
    for item in results:
        for claim in item["acceptance_claims_for_scope_review"]:
            if any(
                value is True
                or (
                    isinstance(value, str)
                    and value.upper() in {"PASS", "ACCEPTED", "ESTABLISHED"}
                )
                for value in claim["claims"].values()
            ):
                positive_claims.append({
                    "path": item["path"], **claim,
                })

    report = {
        "schema_version": "nb21c_action26_twelve_exceptions_C009_v1",
        "status": (
            "ALL_12_FILES_INSPECTED_ADVANCE_WITH_RECORDED_LIMITATIONS"
            if not unresolved else
            "ALL_12_ATTEMPTED_ADVANCE_WITH_UNRESOLVED_READ_LIMITATIONS"
        ),
        "action": 26,
        "source_C008": {
            "path": c008_relative,
            "sha256_file_bytes": c008_hash,
        },
        "policy_population_binding": {
            "path": policy_relative,
            "sha256_file_bytes": policy_expected,
            "count": len(population),
        },
        "exception_count": 12,
        "attempted_count": len(results),
        "successfully_inspected_count": sum(
            item["status"] == "READ_AND_INSPECTED" for item in results
        ),
        "results": results,
        "unresolved_read_limitations": unresolved,
        "positive_acceptance_claims_requiring_scope_review": positive_claims,
        "prior_reference_qualification": source["recorded_qualification"],
        "official_no_answer_metrics": source["official_no_answer_metrics"],
        "decision": {
            "basis": "USER_REQUEST_TO_ADDRESS_12_THEN_ADVANCE",
            "exception_inspection_round_finished": True,
            "next_action": 27,
            "next_action_scope": "CONSUME_EXISTING_SAFETY_EVIDENCE",
            "action_26_completed": False,
            "action_26_status":
                "EVIDENCE_ROUND_FINISHED_FORMAL_ACCEPTANCE_NOT_ESTABLISHED",
            "formal_gate_pass": False,
            "unresolved_items_discarded": False,
            "automatic_new_api_authorized": False,
            "automatic_repeat_scan_planned": False,
        },
        "limitations": [
            "Read success is not semantic correctness or formal acceptance.",
            "Extracted claims require scope/version review before adoption.",
            "No later decision is inferred solely from a cell number.",
            "Historical targeted findings are not population accuracy.",
            "Remaining limitations are carried forward without forced acceptance.",
        ],
        "new_api_calls": 0,
        "new_semantic_judgments": 0,
        "historical_code_executed": False,
        "historical_files_modified": False,
        "checklist_modified": False,
    }

    directory = safe_path(prefix + "NB21C.C009.twelve_exception_review")
    directory.mkdir(parents=True, exist_ok=True)
    body = encode(report)
    digest = sha(body)
    target = directory / f"C009.report.{digest}.json"

    fd, temporary = tempfile.mkstemp(dir=directory, prefix=".C009-")
    try:
        with os.fdopen(fd, "wb") as stream:
            stream.write(body)
            stream.flush()
            os.fsync(stream.fileno())
        try:
            os.link(temporary, target)
        except FileExistsError:
            need(target.read_bytes() == body, "IMMUTABLE_REPORT_CONFLICT")
    finally:
        Path(temporary).unlink(missing_ok=True)

    need(sha(target.read_bytes()) == digest, "REPORT_RELOAD_FAILED")

    print(json.dumps({
        "status": report["status"],
        "attempted_count": len(results),
        "successfully_inspected_count":
            report["successfully_inspected_count"],
        "file_results": [
            {
                "path": item["path"],
                "status": item["status"],
                "unique_test_ids_linked": item["unique_test_ids_linked"],
                "dict_case_id_count_handled":
                    item["dict_case_id_count_handled"],
                "acceptance_claim_count": len(
                    item["acceptance_claims_for_scope_review"]
                ),
                "parse_error_count": len(item["parse_errors"]),
                "error_message": item.get("error_message"),
            }
            for item in results
        ],
        "unresolved_read_limitations": unresolved,
        "positive_acceptance_claim_count": len(positive_claims),
        "positive_acceptance_claim_preview": positive_claims[:12],
        "positive_claim_preview_truncated": len(positive_claims) > 12,
        "decision": report["decision"],
        "new_api_calls": 0,
        "report_path": target.relative_to(root).as_posix(),
        "report_sha256_file_bytes": digest,
        "reload_validation": "PASS",
    }, ensure_ascii=False, indent=2))

    return {
        "status": report["status"],
        "report_path": target.relative_to(root).as_posix(),
        "report_sha256_file_bytes": digest,
        "next_action": 27,
    }


NB21C_C009_RESULT = nb21c_c009_run()

C009: 1/12 — nb20_prepared_blind_span_inputs_v1.jsonl
C009: 2/12 — nb20b_adoption_requests_C179_887c829081a588b0.jsonl
C009: 3/12 — nb20b_compact_adoption_requests_C180_035115738f6fc621.jsonl
C009: 4/12 — nb20b_existing_output_batch_C151_585da4fba202e133.jsonl
C009: 5/12 — C311.plan.json
C009: 6/12 — C192.plan.dbd13741c1654f19b8763aed0a15596bfab0116824405d2ce1ae0e8cafe289e2.json
C009: 7/12 — C195.plan.0075bbc25974f0bf5a639828b22c1fb6028297bf74519ac46cc33342f0453fd1.json
C009: 8/12 — NB21B.C086.corrected_checkpoint_population.0e94fd5a271c3ffa78a24825843bd0bf2ee6a12927f00738d9491450b87bb015.json
C009: 9/12 — NB21B.C088.merged_checkpoints_role_adapter.3a46819187eb7b82ad4d95953e9ab540b6fb84ae33648a914191adf505bf07c6.json
C009: 10/12 — NB21B.C122.bound_verifier_reuse.c8758caf3c9ba94b6ee971a39505ab0269a56ff6c7e02b4cd07a6e6a0c502258.json
C009: 11/12 — NB21B.C163.profile_contract_evidence.5a4fdb9b73b0e7da3c1c76fdd411e4e1fd524b29de8686da787e6e38c9cecaef.json
C009: 12/12 — C005.report.3a9fd93d7e

<div dir="rtl" align="right">

## NB21C.C010 — تکمیل استخراج مستقیم شواهد مرجع برای تمام Test

### هدف

تکمیل بخش فنی تعیین مرجع برای تمام ۱۱۴۹ پرونده،
پیش از هر تصمیم دربارهٔ داوری یا API.

### منابع

- ورودی، پاسخ، Audit و Support اصلی با کنترل هش.
- تمام فایل‌هایی که در C006 به پرونده‌های Test متصل شده‌اند.
- تمام ۱۲ فایلِ استثنای بررسی‌شده در C009.
- فایل‌های دارای اطلاعات سطح فایل دربارهٔ مرجع یا پذیرش در C006.

### خروجی هر پرونده

- سؤال، شواهد اصلی و پاسخ ذخیره‌شده؛
- بررسی تاریخی کامل؛
- فیلدهای صریح مرجع و متن‌های مرتبط با داوری؛
- منشأ، مسیر و محل اطلاعات استخراج‌شده؛
- وضعیت اتصال سؤال و نسخهٔ پاسخ؛
- موارد فاقد شناسه یا دارای اتصال متعارض، در دفتر جداگانه.

هیچ وضعیت تاریخی به برچسب درست تبدیل نمی‌شود.
هیچ پرونده‌ای خودکار برای API ارسال نمی‌شود.
تعداد موارد نیازمند داوری از روی فقدان یک فیلد حدس زده نمی‌شود.

### معیار پایان بخش فنی

تمام فایل‌های تعیین‌شده خوانده شوند، تمام ۱۱۴۹ پرونده بستهٔ ورودی
داوری داشته باشند و خطای خواندن یا اتصال منابع اصلی باقی نماند.

این پایان، مخصوص مجموعهٔ منابع ثبت‌شده است؛
ادعای کشف همهٔ فایل‌های احتمالی در کل پروژه نیست.

</div>

In [14]:
# NB21C.C010 — Direct-source extraction for all 1149 Test cases.
# Replaces prior C010 cells. No API or new semantic judgments.
# May take time: reads original registered sources, not just summaries.

def nb21c_c010_prepare_all():
    import os
    import re
    import gc
    import json
    import hashlib
    import tempfile
    import subprocess
    from pathlib import Path
    from collections import Counter

    def need(ok, message):
        if not ok:
            raise RuntimeError(message)

    def sha(raw):
        return hashlib.sha256(raw).hexdigest()

    def enc(value):
        return json.dumps(
            value, ensure_ascii=False, sort_keys=True,
            separators=(",", ":"), allow_nan=False,
        ).encode("utf-8")

    def parse(raw):
        def pairs(items):
            result = {}
            for key, value in items:
                need(key not in result, "DUPLICATE_JSON_KEY")
                result[key] = value
            return result

        def reject(value):
            raise ValueError("NONSTANDARD_JSON_CONSTANT:" + value)

        return json.loads(
            raw, object_pairs_hook=pairs, parse_constant=reject,
        )

    root = Path(subprocess.run(
        ["git", "--no-optional-locks", "rev-parse", "--show-toplevel"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()).resolve()

    def safe(relative):
        rel = Path(relative)
        need(
            not rel.is_absolute() and ".." not in rel.parts,
            "UNSAFE_PATH",
        )
        path = root
        for part in rel.parts:
            path = path / part
            need(not path.is_symlink(), "SYMLINK_NOT_ALLOWED")
        path.resolve().relative_to(root)
        return path

    def bound(relative, expected):
        raw = safe(relative).read_bytes()
        need(sha(raw) == expected, "HASH_MISMATCH:" + relative)
        return parse(raw)

    prefix = "artifacts/phase2/evaluation_and_ablation/work/outputs/"
    h6 = "8d6059d0ec83417790b31453964c9dcc39aac40122a3b2d1f0d35ab92aa4aa91"
    h9 = "0c608004d538c1b3bd35c33930062fda52939374c733fe2fb1bf57859c94c6a2"

    print("C010: loading registered source inventory...", flush=True)
    c006 = bound(
        prefix + "NB21C.C006.test1149_offline_evidence_inventory/"
        + f"C006.report.{h6}.json", h6,
    )
    c009 = bound(
        prefix + "NB21C.C009.twelve_exception_review/"
        + f"C009.report.{h9}.json", h9,
    )

    population_rows = c006["case_records"]
    ids = {row["candidate_id"] for row in population_rows}
    need(len(population_rows) == len(ids) == 1149, "POPULATION_INVALID")

    inventory = {
        row["path"]: row for row in c006["artifact_inventory"]
    }
    expected_hashes = {
        path: row["sha256_file_bytes"]
        for path, row in inventory.items()
    }

    def explicit_id(obj):
        for field in ("candidate_id", "case_id"):
            value = obj.get(field)
            if isinstance(value, str):
                return value
        # Handle structured case_id without using dicts as set keys.
        value = obj.get("case_id")
        if isinstance(value, dict):
            cid = value.get("candidate_id")
            if isinstance(cid, str):
                return cid
        return None

    # Read core source records directly, preserving full payloads.
    core = {}
    for role in ("inputs", "active", "audit", "support"):
        spec = c006["core_bindings"][role]
        raw = safe(spec["path"]).read_bytes()
        need(sha(raw) == spec["sha256_file_bytes"], "CORE_HASH:" + role)
        index = {}
        for line in raw.splitlines():
            if not line.strip():
                continue
            row = parse(line)
            cid = explicit_id(row)
            need(cid in ids and cid not in index, "CORE_ID:" + role)
            index[cid] = row
        need(set(index) == ids, "CORE_COVERAGE:" + role)
        core[role] = index
        del raw

    case_inputs = {}
    for cid in sorted(ids):
        messages = core["inputs"][cid].get("messages", [])
        users = [
            row for row in messages
            if isinstance(row, dict) and row.get("role") == "user"
        ]
        need(users, "USER_MESSAGE_MISSING:" + cid)
        payload = parse(users[-1]["content"])
        question, evidence = payload.get("question"), payload.get("evidence")
        need(isinstance(question, str), "QUESTION_MISSING:" + cid)
        need(isinstance(evidence, list), "EVIDENCE_MISSING:" + cid)

        active = core["active"][cid]
        response = active.get("response")
        rhash = sha(enc(response))
        need(
            rhash == active.get("response_sha256")
            == core["audit"][cid].get("response_sha256")
            == core["support"][cid].get("response_sha256"),
            "CORE_RESPONSE_BINDING:" + cid,
        )
        case_inputs[cid] = {
            "question": question,
            "query_sha256": sha(question.encode("utf-8")),
            "original_evidence": evidence,
            "original_evidence_sha256": sha(enc(evidence)),
            "active_response": response,
            "active_response_sha256": rhash,
        }

    # Select original files from existing registered links.
    # Include file-level reference/acceptance material and ALL 12 exceptions.
    files = set()
    for case in population_rows:
        for fact in case.get("additional_artifact_facts", []):
            path = fact.get("path")
            if isinstance(path, str):
                files.add(path)

    relevance = re.compile(
        r"reference|answerability|abstention|acceptance|qualification|"
        r"independent|semantic|reviewer|sufficiency", re.I,
    )
    for path, artifact in inventory.items():
        if any(
            relevance.search(str(key))
            for fact in artifact.get("file_level_recorded_facts", [])
            for key in fact.get("recorded_fields", {})
        ):
            files.add(path)

    exception_paths = set()
    for item in c009["results"]:
        path = item["path"]
        exception_paths.add(path)
        files.add(path)
        digest = item.get("sha256_file_bytes")
        need(isinstance(digest, str), "C009_SOURCE_HASH_MISSING:" + path)
        if path in expected_hashes:
            need(expected_hashes[path] == digest, "SOURCE_VERSION_CONFLICT")
        expected_hashes[path] = digest

    need(len(exception_paths) == 12, "EXCEPTION_SET_NOT_12")

    # Directly preserve relevant review content, including dict/list fields.
    # Do not infer expected behavior from prose or historical status.
    relevant_key = re.compile(
        r"reference|answerability|abstention|clarif|sufficien|"
        r"expected.*(?:action|behavior|answer)|review|adjudicat|"
        r"semantic|qualification|independen|acceptance|verdict|"
        r"assessment|reason|rationale|witness|dimension|"
        r"essential_requirement|missing_necessary|requested_operation",
        re.I,
    )
    context_keys = {
        "candidate_id", "case_id", "status", "schema_version",
        "method", "model", "reviewer_identity", "review_cohort",
        "query_sha256", "response_sha256", "active_response_sha256",
        "source_response_sha256", "completion_scope", "metric_scope",
        "requirement_id", "checklist_action_number",
    }

    outdir = safe(prefix + "NB21C.C010.full1149_direct_preparation")
    outdir.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(
        dir=outdir, prefix=".C010-evidence-", suffix=".jsonl",
    )

    counts = Counter()
    linked_counts = Counter()
    unbound_count = 0
    source_results = []

    def walk_document(doc, location, write):
        nonlocal unbound_count
        # Inherit identity and binding context, retaining child overrides.
        stack = [(doc, location, None, {})]
        while stack:
            value, pointer, owner, inherited = stack.pop()
            if isinstance(value, dict):
                local = {
                    key: value[key] for key in context_keys if key in value
                }
                context = {**inherited, **local}
                cid = explicit_id(value)
                if cid is not None:
                    owner = cid if cid in ids else None

                selected = {
                    key: item for key, item in value.items()
                    if relevant_key.search(str(key))
                }
                if selected:
                    target = case_inputs.get(owner)
                    qh = context.get("query_sha256")
                    rhs = [
                        context[key]
                        for key in (
                            "response_sha256", "active_response_sha256",
                            "source_response_sha256",
                        )
                        if isinstance(context.get(key), str)
                    ]
                    binding = "NO_TEST_ID"
                    if target is not None:
                        conflict = (
                            isinstance(qh, str)
                            and qh != target["query_sha256"]
                        ) or any(
                            h != target["active_response_sha256"] for h in rhs
                        )
                        if conflict:
                            binding = "EXPLICIT_VERSION_CONFLICT"
                        elif qh == target["query_sha256"]:
                            binding = "QUESTION_HASH_MATCH"
                        else:
                            binding = "TEST_ID_ONLY_REQUIRES_SCOPE_REVIEW"
                        linked_counts[owner] += 1
                    else:
                        unbound_count += 1

                    write({
                        "pointer": pointer,
                        "candidate_id": owner,
                        "binding_status": binding,
                        "recorded_context": context,
                        "review_content": selected,
                        "new_semantic_judgment": False,
                    })
                    counts[binding] += 1

                for key, child in value.items():
                    if isinstance(child, (dict, list)):
                        key_text = str(key)
                        child_owner = key_text if key_text in ids else owner
                        escaped = key_text.replace("~", "~0").replace("/", "~1")
                        stack.append((
                            child, pointer + "/" + escaped,
                            child_owner, context,
                        ))
            elif isinstance(value, list):
                for number in range(len(value) - 1, -1, -1):
                    child = value[number]
                    if isinstance(child, (dict, list)):
                        stack.append((
                            child, pointer + "/" + str(number),
                            owner, inherited,
                        ))

    try:
        with os.fdopen(fd, "wb") as stream:
            for number, relative in enumerate(sorted(files), 1):
                if number == 1 or number % 250 == 0:
                    print(
                        f"C010: reading original files {number}/{len(files)}",
                        flush=True,
                    )
                result = {"path": relative, "status": "NOT_READ"}
                try:
                    need(relative in expected_hashes, "UNREGISTERED_SOURCE_HASH")
                    path = safe(relative)
                    before = path.stat()
                    digest = hashlib.sha256()
                    doc_count = 0
                    parse_errors = []

                    def write(record):
                        stream.write(enc({
                            "source_path": relative, **record,
                        }) + b"\n")

                    if path.suffix.lower() == ".jsonl":
                        with path.open("rb") as handle:
                            for line_number, line in enumerate(handle, 1):
                                digest.update(line)
                                if not line.strip():
                                    continue
                                try:
                                    doc = parse(line)
                                except Exception as error:
                                    parse_errors.append({
                                        "line": line_number,
                                        "type": type(error).__name__,
                                        "message": str(error)[:300],
                                    })
                                    continue
                                walk_document(doc, f"line:{line_number}", write)
                                doc_count += 1
                                del doc
                    else:
                        raw = path.read_bytes()
                        digest.update(raw)
                        doc = parse(raw)
                        del raw
                        walk_document(doc, "", write)
                        doc_count = 1
                        del doc

                    after = path.stat()
                    need(
                        before.st_size == after.st_size
                        and before.st_mtime_ns == after.st_mtime_ns,
                        "SOURCE_CHANGED_DURING_READ",
                    )
                    need(
                        digest.hexdigest() == expected_hashes[relative],
                        "SOURCE_HASH_MISMATCH",
                    )
                    result.update({
                        "status": (
                            "READ_WITH_PARSE_ERRORS"
                            if parse_errors else "READ_AND_EXTRACTED"
                        ),
                        "sha256_file_bytes": digest.hexdigest(),
                        "document_count": doc_count,
                        "parse_errors": parse_errors,
                    })
                except Exception as error:
                    result.update({
                        "status": "READ_FAILED",
                        "error_type": type(error).__name__,
                        "error": str(error)[:500],
                    })
                source_results.append(result)
                if number % 100 == 0:
                    gc.collect()

            stream.flush()
            os.fsync(stream.fileno())

        def file_sha(path):
            digest = hashlib.sha256()
            with Path(path).open("rb") as handle:
                for block in iter(lambda: handle.read(4 * 1024 * 1024), b""):
                    digest.update(block)
            return digest.hexdigest()

        evidence_hash = file_sha(temporary)
        evidence_path = outdir / f"C010.evidence.{evidence_hash}.jsonl"
        try:
            os.link(temporary, evidence_path)
        except FileExistsError:
            need(file_sha(evidence_path) == evidence_hash, "EVIDENCE_CONFLICT")
    finally:
        Path(temporary).unlink(missing_ok=True)

    # Full original material, available for later judgment without rereading
    # the large source inventory or regenerating answers.
    packet_fd, packet_temp = tempfile.mkstemp(
        dir=outdir, prefix=".C010-packets-", suffix=".jsonl",
    )
    try:
        with os.fdopen(packet_fd, "wb") as handle:
            for cid in sorted(ids):
                packet = {
                    "candidate_id": cid,
                    **case_inputs[cid],
                    "historical_audit": core["audit"][cid],
                    "historical_support": core["support"][cid],
                    "direct_review_record_count": linked_counts[cid],
                    "direct_review_evidence_file": evidence_path.name,
                    "reference_label_newly_inferred": None,
                    "api_dispatch_authorized": False,
                }
                handle.write(enc(packet) + b"\n")
            handle.flush()
            os.fsync(handle.fileno())

        packet_hash = file_sha(packet_temp)
        packet_path = outdir / f"C010.packets1149.{packet_hash}.jsonl"
        try:
            os.link(packet_temp, packet_path)
        except FileExistsError:
            need(file_sha(packet_path) == packet_hash, "PACKET_CONFLICT")
    finally:
        Path(packet_temp).unlink(missing_ok=True)

    failures = [
        row for row in source_results
        if row["status"] != "READ_AND_EXTRACTED"
    ]
    ready = not failures and len(case_inputs) == 1149
    report = {
        "schema_version": "nb21c_full1149_direct_preparation_v1",
        "status": (
            "TECHNICAL_EXTRACTION_COMPLETE_READY_FOR_REFERENCE_REVIEW"
            if ready else
            "TECHNICAL_EXTRACTION_HAS_EXPLICIT_FAILURES"
        ),
        "population_count": 1149,
        "packet_count": len(case_inputs),
        "registered_source_count": len(files),
        "successfully_read_source_count": len(files) - len(failures),
        "all_12_exception_sources_included": exception_paths <= files,
        "binding_record_counts": dict(counts),
        "cases_with_direct_review_records": len(linked_counts),
        "unbound_review_record_count": unbound_count,
        "source_results": source_results,
        "failures": failures,
        "evidence_file": {
            "path": evidence_path.relative_to(root).as_posix(),
            "sha256_file_bytes": evidence_hash,
        },
        "population_packets": {
            "path": packet_path.relative_to(root).as_posix(),
            "sha256_file_bytes": packet_hash,
        },
        "technical_preparation_complete": ready,
        "semantic_reference_selection_complete": False,
        "new_api_calls": 0,
        "new_semantic_judgments": 0,
        "automatic_api_queue_created": False,
        "checklist_modified": False,
        "limits": [
            "Complete for the registered source set, not every possible project file.",
            "Prose under string fields is preserved, not executed or guessed.",
            "Nested JSON embedded inside strings is preserved as text.",
            "Repeated source material is not independent voting evidence.",
            "Unbound and version-conflicting records remain available for review.",
            "Read or hash failures prevent technical-ready status.",
            "No count here establishes how many new API judgments are necessary.",
        ],
    }

    raw = enc(report)
    digest = sha(raw)
    report_path = outdir / f"C010.preparation.{digest}.json"
    if report_path.exists():
        need(report_path.read_bytes() == raw, "REPORT_CONFLICT")
    else:
        with report_path.open("xb") as handle:
            handle.write(raw)
            handle.flush()
            os.fsync(handle.fileno())
    need(sha(report_path.read_bytes()) == digest, "REPORT_RELOAD_FAILED")

    summary = {
        key: report[key] for key in (
            "status", "population_count", "packet_count",
            "registered_source_count", "successfully_read_source_count",
            "all_12_exception_sources_included",
            "binding_record_counts", "cases_with_direct_review_records",
            "unbound_review_record_count", "technical_preparation_complete",
            "semantic_reference_selection_complete",
            "new_api_calls", "automatic_api_queue_created",
        )
    }
    summary.update({
        "failures": failures,
        "evidence_file": report["evidence_file"],
        "population_packets": report["population_packets"],
        "report_path": report_path.relative_to(root).as_posix(),
        "report_sha256_file_bytes": digest,
        "reload_validation": "PASS",
    })
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary


NB21C_C010_PREPARATION = nb21c_c010_prepare_all()

C010: loading registered source inventory...
C010: reading original files 1/23095
C010: reading original files 250/23095
C010: reading original files 500/23095
C010: reading original files 750/23095
C010: reading original files 1000/23095
C010: reading original files 1250/23095
C010: reading original files 1500/23095
C010: reading original files 1750/23095
C010: reading original files 2000/23095
C010: reading original files 2250/23095
C010: reading original files 2500/23095
C010: reading original files 2750/23095
C010: reading original files 3000/23095
C010: reading original files 3250/23095
C010: reading original files 3500/23095
C010: reading original files 3750/23095
C010: reading original files 4000/23095
C010: reading original files 4250/23095
C010: reading original files 4500/23095
C010: reading original files 4750/23095
C010: reading original files 5000/23095
C010: reading original files 5250/23095
C010: reading original files 5500/23095
C010: reading original files 5750/23095
C

<div dir="rtl" align="right">

## NB21C.C011 — انتشار خواندنیِ شواهد موجود برای جلوگیری از API تکراری

### هدف

قابل‌خواندن‌کردن منابع اصلیِ داوری و اتصال آن‌ها در مخزن،
بدون تولید دوبارهٔ نتایج یا انتقال خروجی بسیار بزرگ استخراج.

### منابع

- رجیستری نهایی C371 شامل تمام ۱۱۴۹ بررسی ثبت‌شده.
- گزارش C392 دربارهٔ اتصال بررسی‌ها به پاسخ و شواهد.
- مرجع پیشنهادی رفتار C197.
- گزارش پایان استخراج مستقیم C010.

### روش

هش منابع کنترل می‌شود. محتوا به رکوردهای مستقل تقسیم می‌شود؛
هیچ متن داوری کوتاه یا حذف نمی‌شود.
برای اشیای بزرگ، تقسیم تا سطح فرزند ادامه پیدا می‌کند
و مسیر JSON هر قطعه حفظ می‌شود.

### خروجی

بخش‌های JSONL کوچک، همراه Manifest، شمار رکوردها و هش منابع.
این بسته برای بررسی بازاستفادهٔ داوری‌هاست؛
پذیرش خودکار مرجع یا صف API ایجاد نمی‌کند.

</div>

In [15]:
# NB21C.C011 — Export exact existing evidence into readable small parts.
# No API, broad scan, new judgments, git commit, or checklist change.

def nb21c_c011_export():
    import os
    import json
    import hashlib
    import subprocess
    from pathlib import Path

    def need(ok, message):
        if not ok:
            raise RuntimeError(message)

    def sha(raw):
        return hashlib.sha256(raw).hexdigest()

    def enc(value):
        return json.dumps(
            value, ensure_ascii=False, sort_keys=True,
            separators=(",", ":"), allow_nan=False,
        ).encode("utf-8")

    root = Path(subprocess.run(
        ["git", "--no-optional-locks", "rev-parse", "--show-toplevel"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()).resolve()

    def safe(relative):
        rel = Path(relative)
        need(
            not rel.is_absolute() and ".." not in rel.parts,
            "UNSAFE_PATH",
        )
        path = root
        for component in rel.parts:
            path = path / component
            need(not path.is_symlink(), "SYMLINK_NOT_ALLOWED")
        path.resolve().relative_to(root)
        return path

    a = "artifacts/phase2/answerability_safety_confidence/work/outputs/"
    e = "artifacts/phase2/evaluation_and_ablation/work/outputs/"

    sources = [
        {
            "role": "C371_FINAL_REVIEW_REGISTRY",
            "path": a + "nb20_completed_scoped_registry_C371_v1.jsonl",
            "sha256":
                "70d6767991eac4081fcb1fe4aad71162ed69b24f794df5d86df1e1b5ac20c269",
            "format": "jsonl",
        },
        {
            "role": "C392_REVIEW_CONTEXT_BINDING",
            "path": a + "nb20c_action23_review_context_binding_C392_v1/"
                "C392.binding."
                "d73c940bfb84f7e17b82b92c9707add39349aa03bc6f2d36dea5bc9e38af3af3.json",
            "sha256":
                "d73c940bfb84f7e17b82b92c9707add39349aa03bc6f2d36dea5bc9e38af3af3",
            "format": "json",
        },
        {
            "role": "C197_BEHAVIOR_REFERENCE",
            "path": a + "nb20c_model_reference_registry_C197_v1/"
                "C197.references."
                "1a33dbdd0cb352794f7f3fa6228f0279f4a1d832cb573bea9e802211d2dcfc12.json",
            "sha256":
                "1a33dbdd0cb352794f7f3fa6228f0279f4a1d832cb573bea9e802211d2dcfc12",
            "format": "json",
        },
        {
            "role": "C010_TECHNICAL_PREPARATION",
            "path": e + "NB21C.C010.full1149_direct_preparation/"
                "C010.preparation."
                "d3e90166ecca477a91c114ff2e06dbb2a06034104e3b1639a7e319da98063ab0.json",
            "sha256":
                "d3e90166ecca477a91c114ff2e06dbb2a06034104e3b1639a7e319da98063ab0",
            "format": "json",
        },
    ]

    # A source-hash-bound directory avoids overwriting a different export.
    bundle_id = sha(enc(sources))[:16]
    relative_out = (
        "notebooks/data_load_read/"
        + "NB21C_C011_existing_review_sources_" + bundle_id
    )
    out = safe(relative_out)
    out.mkdir(parents=True, exist_ok=True)

    def save_exact(path, raw):
        if path.exists():
            need(path.read_bytes() == raw, "EXISTING_EXPORT_DIFFERS")
            return
        # A partial file is never used as a valid final export.
        temporary = path.with_name(path.name + ".temporary")
        try:
            with temporary.open("xb") as stream:
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())
            os.link(temporary, path)
        finally:
            temporary.unlink(missing_ok=True)
        need(path.read_bytes() == raw, "EXPORT_RELOAD_FAILED")

    max_record_bytes = 160 * 1024
    max_part_bytes = 600 * 1024
    manifest_sources = []
    parts = []
    pending = []
    pending_size = 0

    def flush():
        nonlocal pending, pending_size
        if not pending:
            return
        raw = b"".join(pending)
        name = f"part_{len(parts) + 1:04d}.jsonl"
        save_exact(out / name, raw)
        parts.append({
            "file": name,
            "records": len(pending),
            "bytes": len(raw),
            "sha256_file_bytes": sha(raw),
        })
        pending = []
        pending_size = 0

    def emit(record):
        nonlocal pending_size
        raw = enc(record) + b"\n"
        if pending and pending_size + len(raw) > max_part_bytes:
            flush()
        pending.append(raw)
        pending_size += len(raw)

    def export_value(role, pointer, value, source_line=None):
        record = {
            "source_role": role,
            "source_line": source_line,
            "json_pointer": pointer,
            "encoding": "VALUE",
            "value": value,
        }
        if len(enc(record)) <= max_record_bytes:
            emit(record)
            return

        if isinstance(value, dict):
            emit({
                "source_role": role, "source_line": source_line,
                "json_pointer": pointer, "encoding": "OBJECT_CHILDREN",
                "keys": list(value),
            })
            for key, child in value.items():
                escaped = str(key).replace("~", "~0").replace("/", "~1")
                export_value(
                    role, pointer + "/" + escaped, child, source_line,
                )
        elif isinstance(value, list):
            emit({
                "source_role": role, "source_line": source_line,
                "json_pointer": pointer, "encoding": "ARRAY_CHILDREN",
                "length": len(value),
            })
            for index, child in enumerate(value):
                export_value(
                    role, pointer + "/" + str(index), child, source_line,
                )
        elif isinstance(value, str):
            # Preserve even unusually long strings without truncation.
            chunks = [
                value[start:start + 20000]
                for start in range(0, len(value), 20000)
            ]
            for index, chunk in enumerate(chunks):
                emit({
                    "source_role": role, "source_line": source_line,
                    "json_pointer": pointer, "encoding": "STRING_CHUNK",
                    "chunk_index": index, "chunk_count": len(chunks),
                    "value": chunk,
                })
        else:
            raise RuntimeError("UNSUPPORTED_OVERSIZED_SCALAR")

    for spec in sources:
        print("C011: exporting " + spec["role"], flush=True)
        raw = safe(spec["path"]).read_bytes()
        need(sha(raw) == spec["sha256"], "SOURCE_HASH_MISMATCH:" + spec["role"])

        count = 0
        candidate_ids = set()
        if spec["format"] == "jsonl":
            for line_number, line in enumerate(raw.splitlines(), 1):
                if not line.strip():
                    continue
                row = json.loads(line)
                cid = row.get("candidate_id")
                need(
                    isinstance(cid, str) and cid not in candidate_ids,
                    "REGISTRY_ID_INVALID_OR_DUPLICATED",
                )
                candidate_ids.add(cid)
                export_value(spec["role"], "", row, line_number)
                count += 1
            need(count == 1149, "C371_REGISTRY_NOT_1149")
        else:
            doc = json.loads(raw)
            export_value(spec["role"], "", doc)
            count = 1

        manifest_sources.append({
            **spec,
            "bytes": len(raw),
            "source_record_count": count,
            "unique_candidate_count": (
                len(candidate_ids) if candidate_ids else None
            ),
        })
        del raw
        flush()

    manifest = {
        "schema_version": "nb21c_readable_review_sources_C011_v1",
        "status": "EXACT_EXISTING_SOURCES_EXPORTED",
        "sources": manifest_sources,
        "parts": parts,
        "part_count": len(parts),
        "content_truncated": False,
        "new_api_calls": 0,
        "new_semantic_judgments": 0,
        "purpose": "Check prior review reuse before considering new API work",
        "note":
            "Read JSON pointers and source roles; historical criticality "
            "labels are not automatically answerability labels.",
    }
    save_exact(out / "manifest.json", enc(manifest))

    print(json.dumps({
        "status": manifest["status"],
        "registry_case_count": 1149,
        "part_count": len(parts),
        "total_megabytes": round(
            sum(part["bytes"] for part in parts) / (1024 * 1024), 2
        ),
        "output_directory": relative_out,
        "manifest_path": relative_out + "/manifest.json",
        "content_truncated": False,
        "new_api_calls": 0,
        "next_step":
            "Commit and push this output directory only, then provide its path.",
    }, ensure_ascii=False, indent=2))

    return relative_out


NB21C_C011_EXPORT_PATH = nb21c_c011_export()

C011: exporting C371_FINAL_REVIEW_REGISTRY
C011: exporting C392_REVIEW_CONTEXT_BINDING
C011: exporting C197_BEHAVIOR_REFERENCE
C011: exporting C010_TECHNICAL_PREPARATION
{
  "status": "EXACT_EXISTING_SOURCES_EXPORTED",
  "registry_case_count": 1149,
  "part_count": 74,
  "total_megabytes": 40.95,
  "output_directory": "notebooks/data_load_read/NB21C_C011_existing_review_sources_4388cbff8cda586f",
  "manifest_path": "notebooks/data_load_read/NB21C_C011_existing_review_sources_4388cbff8cda586f/manifest.json",
  "content_truncated": false,
  "new_api_calls": 0,
  "next_step": "Commit and push this output directory only, then provide its path."
}
